# ⚖️ 23 · Model selection: bias, variance, cross-validation and the limits of prediction
### Under- and overfitting, the bias–variance decomposition derived and verified, k-fold and leave-one-out cross-validation from scratch, selection bias, regularization paths, BIC, the Bayes error rate, and an honest leukaemia classifier with nested cross-validation

**BiologyToML** · Level 1 · 02-620 core · Part E — Bayesian learning, neural networks and model selection · Module 23 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/23_model_selection.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (SVM, lasso and Gaussian-mixture fits only; every cross-validation loop is written by hand) |
| **Prerequisites** | Modules 06 (linear regression), 07 (ridge/lasso), 08 (k-NN, naïve Bayes) and 09 (logistic regression); Modules 11 (SVM), 13 (mixtures) and 22 (neural networks) help for single sections |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Model selection: parameter estimation vs model selection, under- and overfitting, the bias–variance trade-off and its decomposition (two proofs), cross-validation (hold-out, k-fold, LOOCV), regularization with λ chosen by CV, the Bayesian information criterion, the Bayes error rate, and the leukaemia case study (k-NN, Gaussian classifier, logistic regression) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data in this notebook are SIMULATED by code** (labelled in every plot): simulated look-alikes of the
> Golub et al. (1999) leukaemia expression data (72 patients × 3,571 genes), the Stamey et al. (1989) prostate-cancer
> study (97 men) and the Wisconsin diagnostic breast-cancer data (569 tumours), plus a circadian gene-expression time
> course, GWAS-like genotypes, pure-noise "expression" matrices, repeated biopsies and Gaussian classes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Distinguish** parameter estimation from model selection and **list** the model-selection choices of every 02-620 method — including the SVM's, which short summaries often list as "none".
2. **Explain** underfitting and overfitting with training and test error, **define** bias and variance, and **estimate** both by repeating a fit on hundreds of simulated training sets.
3. **Derive** the decomposition $\mathbb E[(y-\hat f)^2]=\text{Bias}^2+\text{Variance}+\sigma^2$ step by step (two different proofs) and **verify** every term numerically.
4. **Implement** hold-out validation, k-fold cross-validation and LOOCV from scratch; **evaluate** how k changes cost and the bias/variance of the estimate, and **recognise** when grouped (non-independent) samples break CV.
5. **Demonstrate** selection bias (Ambroise & McLachlan 2002) on pure noise and on Golub-like leukaemia data, and **perform** feature selection correctly inside the folds.
6. **Choose** a regularization parameter by cross-validation (lasso on prostate cancer) and **relate** bias² + variance to the test error along λ.
7. **Compute** BIC to choose a polynomial degree and a number of mixture components, and **compare** it with CV.
8. **Compute** the Bayes error rate of two overlapping classes, **show** that k-NN and logistic regression cannot beat it, and **carry out** the Golub k-NN / Gaussian / logistic-regression case study with nested cross-validation.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, data | 5 min |
| 1 · Parameter estimation vs model selection | what 'the model M' means for each method | 10 min |
| 2 · Training vs test error | a circadian time course fitted with degree 1, 3 and 7 | 15 min |
| 3 · Bias and variance | 200 training sets; feature subsets, HMM states, BN edges | 10 min |
| 4 · The bias–variance decomposition | derive it twice, verify every term, curves vs complexity | 20 min |
| 5 · Cross-validation from scratch | hold-out, k-fold, LOOCV, choosing k, grouped data, tuning an SVM | 25 min |
| 6 · The cardinal rule and selection bias | selecting genes outside vs inside the folds | 20 min |
| 7 · Regularization chosen by CV | lasso on prostate; bias² + variance along λ | 15 min |
| 8 · Bayesian information criterion | polynomial degree and mixture components | 15 min |
| 9 · The Bayes error rate | the floor no classifier can beat | 15 min |
| 10 · Case study: Golub leukaemia | three classifiers, nested cross-validation | 20 min |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint, discussion, going further, lab note | 10 min |

## 🧫 The biological question

In 1999 Golub and colleagues showed that the expression of a few dozen genes, measured on microarrays, distinguishes
acute lymphoblastic leukaemia (ALL) from acute myeloid leukaemia (AML) — two diseases that need different treatments.
Their data set, 72 patients and thousands of genes, became a benchmark for a whole generation of classifiers, and many
papers reported error rates of zero or close to it. (The repository stores no data, so this notebook uses a
**simulated** data set shaped like Golub's — same size, same classes, genes named after the markers of the original
study; simulated data also tell us which genes truly differ, a ground truth no real study has.)

Three years later Ambroise & McLachlan (2002) pointed out that several of those error rates were **too good to be
true**: the genes had been chosen using *all* the samples, and only then was the classifier "cross-validated". When the
gene selection was repeated inside every cross-validation fold, the error rates went up. The lesson reaches far beyond
leukaemia: whenever we have many more features than patients (genomics, proteomics, imaging), the way we *choose* a
model — which genes, how many neighbours, how strong a penalty, how many hidden units — can fool us more easily than
the model itself.

This last lecture of 02-620 is about exactly that: **model selection**. How flexible should a model be (bias and
variance)? How do we estimate its performance on patients we have not seen (cross-validation, BIC)? What mistakes make
the estimate dishonest (using the test set, selection bias, correlated samples)? And what is the best performance any
method could possibly reach (the Bayes error rate)? We finish with the lecture's case study — three leukaemia classifiers — done
honestly, with **nested** cross-validation.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the simulated leukaemia, prostate and tumour data sets, `standardize`, `sigmoid`, `accuracy`, the genotype and trait simulators, and the signalling network of Module 17 (only to count parameters). Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_prostate_clinical(seed: int = 1989) -> pd.DataFrame:
    """Synthetic clinical data shaped like the prostate-cancer study of Stamey et al. (1989).

    97 men about to undergo radical prostatectomy; 8 clinical measurements — log cancer volume
    (``lcavol``), log prostate weight (``lweight``), ``age``, log benign prostatic hyperplasia
    amount (``lbph``), seminal-vesicle invasion (``svi``, 0/1), log capsular penetration
    (``lcp``), Gleason score (``gleason``) and % Gleason scores 4 or 5 (``pgg45``) — and the
    outcome ``lpsa`` = log(prostate-specific antigen).  Marginals and correlations mimic the
    original data (``lbph`` and ``lcp`` have a floor at log 0.25 = −1.386); ``lpsa`` is generated
    from the standardised predictors with the least-squares coefficients reported for the
    original study by Hastie, Tibshirani & Friedman (2009, Table 3.2) plus noise (sd 0.71).
    The boolean ``train`` marks a 67/30 training/test split.
    """
    rng = np.random.default_rng(seed)
    n = 97
    names = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
    R = np.array([[1.00, 0.29, 0.19, 0.01, 0.56, 0.66, 0.49, 0.50],         # rank correlations of the
                  [0.29, 1.00, 0.40, 0.49, 0.17, 0.18, 0.14, 0.17],         # original study (rounded)
                  [0.19, 0.40, 1.00, 0.34, 0.13, 0.15, 0.25, 0.27],
                  [0.01, 0.49, 0.34, 1.00, -0.09, 0.02, 0.11, 0.10],
                  [0.56, 0.17, 0.13, -0.09, 1.00, 0.62, 0.38, 0.48],
                  [0.66, 0.18, 0.15, 0.02, 0.62, 1.00, 0.58, 0.66],
                  [0.49, 0.14, 0.25, 0.11, 0.38, 0.58, 1.00, 0.87],
                  [0.50, 0.17, 0.27, 0.10, 0.48, 0.66, 0.87, 1.00]])
    w, V = np.linalg.eigh(R)
    R = V @ np.diag(np.clip(w, 0.02, None)) @ V.T                        # make it positive definite
    Z = rng.multivariate_normal(np.zeros(8), R, n)
    from scipy.stats import norm
    U = norm.cdf(Z)
    floor = -1.38629
    df = pd.DataFrame({
        "lcavol": 1.35 + 1.18 * Z[:, 0],
        "lweight": 3.63 + 0.43 * Z[:, 1],
        "age": np.clip(np.round(63.9 + 7.4 * Z[:, 2]), 41, 79).astype(int),
        "lbph": np.where(U[:, 3] < 0.44, floor, np.clip(1.28 + 0.77 * norm.ppf((U[:, 3] - 0.44) / 0.56), -0.8, 2.4)),
        "svi": (U[:, 4] > 0.78).astype(int),
        "lcp": np.where(U[:, 5] < 0.46, floor, np.clip(0.87 + 1.13 * norm.ppf((U[:, 5] - 0.46) / 0.54), -0.8, 2.9)),
        "gleason": np.searchsorted([0.36, 0.94, 0.95], U[:, 6]) + 6,
    })
    df["pgg45"] = np.where(df["gleason"] == 6, 0, np.clip(5 * np.round((38 + 27 * Z[:, 7]) / 5), 5, 100)).astype(int)
    zs = (df - df.mean()) / df.std()
    coef = dict(lcavol=0.68, lweight=0.26, age=-0.14, lbph=0.21, svi=0.31, lcp=-0.29, gleason=-0.02, pgg45=0.27)
    df["lpsa"] = 2.47 + sum(c * zs[k] for k, c in coef.items()) + rng.normal(0, 0.71, n)
    for col in ("lcavol", "lweight", "lbph", "lcp", "lpsa"):
        df[col] = np.round(df[col], 5)
    df["train"] = rng.permutation(np.r_[np.ones(67, bool), np.zeros(30, bool)])
    return df[names + ["lpsa", "train"]]


def simulate_tumor_features(seed: int = 1995) -> dict:
    """Synthetic breast-tumour nuclear features shaped like the Wisconsin diagnostic data set.

    569 fine-needle aspirates (212 malignant, 357 benign).  For 10 properties of the cell nuclei in
    each image — radius, texture, perimeter, area, smoothness, compactness, concavity, concave
    points, symmetry, fractal dimension — three summaries are given: the mean, the standard error
    ("error") and the largest value ("worst"), i.e. 30 features.  Geometry ties them together
    (perimeter ≈ 2π·radius, area ≈ π·radius²) and a shared shape-irregularity factor links
    compactness, concavity and concave points, so many features are strongly correlated.

    Returns a dict with ``X`` (DataFrame, 569 × 30), ``y`` (1 = malignant, 0 = benign),
    ``target_names`` and ``source``.
    """
    rng = np.random.default_rng(seed)
    y = rng.permutation(np.r_[np.ones(212, int), np.zeros(357, int)])
    n = len(y)
    m = y == 1
    radius = np.where(m, rng.normal(17.3, 3.0, n), rng.normal(12.2, 1.8, n)).clip(6.5, 28)
    texture = np.where(m, rng.normal(21.5, 3.8, n), rng.normal(17.9, 4.0, n)).clip(9, 40)
    irregular = rng.normal(0, 1, n) + np.where(m, 2.3, 0.0)               # shared shape factor
    smooth = (0.092 + 0.008 * m + 0.012 * rng.normal(0, 1, n) + 0.004 * irregular).clip(0.05, 0.17)
    compact = np.exp(-2.5 + 0.38 * irregular + 0.2 * rng.normal(0, 1, n))
    concav = np.exp(-3.4 + 0.75 * irregular + 0.35 * rng.normal(0, 1, n))
    cpoints = np.exp(-3.8 + 0.6 * irregular + 0.25 * rng.normal(0, 1, n))
    symmetry = (0.175 + 0.008 * irregular + 0.025 * rng.normal(0, 1, n)).clip(0.1, 0.3)
    fractal = (0.063 + 0.002 * irregular - 0.0004 * (radius - 14) + 0.006 * rng.normal(0, 1, n)).clip(0.045, 0.1)
    perimeter = 2 * np.pi * radius * (1 + 0.08 * compact / compact.mean()) + rng.normal(0, 1.5, n)
    area = np.pi * radius ** 2 * (0.97 + 0.03 * rng.normal(0, 1, n))
    means = dict(radius=radius, texture=texture, perimeter=perimeter, area=area, smoothness=smooth,
                 compactness=compact, concavity=concav, **{"concave points": cpoints}, symmetry=symmetry,
                 **{"fractal dimension": fractal})
    cols = {}
    spread = np.exp(rng.normal(0, 0.35, n)) * (1 + 0.6 * m)              # heterogeneity of the nuclei
    for name, v in means.items():
        cols[f"mean {name}"] = v
    for name, v in means.items():
        cols[f"{name} error"] = v * 0.07 * spread * np.exp(rng.normal(0, 0.25, n))
    for name, v in means.items():
        cols[f"worst {name}"] = v * (1 + 0.18 * spread * np.exp(rng.normal(0, 0.2, n)))
    X = pd.DataFrame(cols).round(5)
    return dict(X=X, y=y, target_names=np.array(["benign", "malignant"], dtype=object),
                source="SIMULATED tumour features (shaped like the Wisconsin diagnostic breast-cancer data)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def simulate_quantitative_trait(rng: np.random.Generator, G: np.ndarray, n_causal: int = 5,
                                h2: float = 0.5, causal=None):
    """Simulate a quantitative phenotype y = G·β + ε with ``n_causal`` causal SNPs.

    The genetic part is scaled so that it explains a fraction ``h2`` (the heritability) of
    the phenotypic variance.  Returns ``(y, beta, causal_idx)``.
    """
    N, J = G.shape
    causal_idx = np.sort(rng.choice(J, n_causal, replace=False) if causal is None else np.asarray(causal))
    beta = np.zeros(J)
    beta[causal_idx] = rng.choice([-1, 1], len(causal_idx)) * rng.uniform(0.5, 1.0, len(causal_idx))
    g = G @ beta
    scale = np.sqrt(h2 / g.var()) if g.var() > 0 else 0.0
    beta *= scale
    y = G @ beta + rng.normal(0, np.sqrt(1 - h2), N)
    return y, beta, causal_idx


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_prostate_clinical', 'simulate_tumor_features', 'simulate_genotypes', 'simulate_quantitative_trait', 'sigmoid', 'standardize', 'accuracy']))

---
## 1 · Parameter estimation vs model selection

> 📍 **Where we are.** Every module so far fitted the parameters of a *given* model: regression weights, naïve-Bayes
> probabilities, network weights. This module asks how to choose the model itself. The lecture's plan: what model
> selection means; overfitting and the bias–variance trade-off; strategies in practice (cross-validation,
> regularization, BIC); the theory behind bias and variance; and the fundamental accuracy of the optimal classifier
> (the Bayes error rate).

**Learning = parameter estimation, assuming a known model $M$.** For a probabilistic model we have three
ways to use the data $D$ ([Module 05](05_likelihood_mle_map.ipynb), [Module 21](21_bayesian_learning_mcmc.ipynb)):

$$\text{MLE: }\max_\theta\ \log P(D\mid\theta,M),\qquad \text{MAP: }\max_\theta\ \log P(\theta\mid D,M),\qquad
\text{fully Bayesian: } P(\theta\mid D,M).$$

*In words:* MLE picks the parameters under which the observed data are most probable; MAP also weighs in a prior;
the fully Bayesian approach keeps the whole posterior distribution of the parameters. All three take the model $M$ —
which features, how many clusters, which graph — as given. (Non-probabilistic models, such as k-NN or an SVM, also
have parameters or settings fitted for a given $M$.) **Model selection asks: how do we learn $M$?**

Here is what "$M$" means for each method of the course:

| Method | Model-selection choice(s) | Note |
|---|---|---|
| Linear / logistic regression | which features; penalty λ (ridge, lasso) | Modules 06, 07, 09 |
| k-nearest neighbours | **number of neighbours** $k$; distance; feature scaling | $k$ counts *neighbours* — k-NN does not cluster, so do not confuse it with the $K$ of k-means |
| Decision tree / random forest | tree size (depth, minimum leaf size, pruning); features tried per split | Module 10 |
| **SVM** | **$C$, kernel type and its parameters (e.g. $\gamma$ of the RBF kernel, polynomial degree), class weights, feature scaling** | sometimes listed as "NA" (nothing to select) — an oversimplification; Section 5.6 shows the accuracy swinging from 63 % to 95 % with $C$ and $\gamma$ |
| Bayesian networks / Gaussian graphical models | graph structure (score penalty, graphical-lasso λ) | Modules 17, 18 |
| k-means, mixture models | number of clusters $K$ (and covariance type) | Modules 12, 13 |
| Hidden Markov models | number of hidden states | Modules 19, 20 |
| Neural networks | architecture (layers, units), weight decay, learning rate, epochs | Module 22 |

⚠️ **Why "SVM: NA" is misleading.** The SVM's training problem is convex and has a unique solution *for fixed* $C$ and
kernel ([Module 11](11_support_vector_machines.ipynb)), which is what such a summary means: there is no local-optimum
issue. But choosing $C$ and the kernel is model selection in every sense of this lecture, and getting it wrong is costly.

✏️ **Your turn.** The table does not list naïve Bayes ([Module 08](08_knn_naive_bayes.ipynb)) or PCA
([Module 14](14_pca_dimensionality_reduction.ipynb)). What are *their* model-selection choices? (Hints: which features,
which distribution per feature, the pseudo-count of Laplace smoothing; the number of principal components.)

---
## 2 · Training error and test error: under- and overfitting

> 📍 **Where we are.** *What is a good model?* The classic illustration fits three curves to the same noisy points — a
> straight line, a moderately wiggly curve and a very wiggly one — and asks which has the lowest **training** error (on
> the known data) and which the lowest **testing** error (on new data). We do that experiment with a biological time
> course.

**The data.** The mRNA level of a clock-controlled gene rises and falls once per day. We simulate a measurement every
2.7 hours over one day (10 time points, as in a typical time-course experiment), with true log-expression
$f(t)=\sin(2\pi t)$ ($t$ = fraction of a day) plus measurement noise of standard deviation $\sigma=0.3$:

$$y_i=f(t_i)+\epsilon_i,\qquad \epsilon_i\sim\mathcal N(0,\sigma^2).$$

*In words:* each measurement is the true expression at that time plus an independent random error with mean 0 and
spread 0.3. The models are polynomials of degree $d$: $\hat f(t)=w_0+w_1t+\dots+w_dt^d$, fitted by least squares
([Module 06](06_linear_regression_gwas.ipynb)). $d=1$ is the simple model $y=w_0+wx+\epsilon$; $d=7$ a complex one.

🧮 **Math refresher — mean squared error.** For predictions $\hat y_i$ of $n$ values $y_i$,
$\mathrm{MSE}=\frac1n\sum_{i=1}^n(y_i-\hat y_i)^2$, the average squared miss. The **training error** is the MSE on the
points used for fitting; the **test error** is the MSE on *new* measurements the model has never seen.

In [ ]:
# ── 2.1 The SIMULATED circadian gene and polynomial least squares ──
SIGMA = 0.3
T_FIXED = np.linspace(0, 1, 10)                     # 10 time points from 0 h to 24 h (fraction of a day)


def circadian(t):
    """True mean log-expression of the SIMULATED clock-controlled gene at time t (fraction of a day)."""
    return np.sin(2 * np.pi * t)


def poly_design(t, degree):
    """Columns 1, t, t², …, t^degree (the polynomial regression design matrix)."""
    return np.vander(np.asarray(t, float), degree + 1, increasing=True)


def fit_poly(t, y, degree):
    """Least-squares polynomial weights w0 … w_degree (y may be a matrix: one column per data set)."""
    return np.linalg.lstsq(poly_design(t, degree), y, rcond=None)[0]


def predict_poly(w, t):
    """Evaluate the fitted polynomial(s) at times t."""
    return poly_design(t, w.shape[0] - 1) @ w


print("time points (h):", np.round(T_FIXED * 24, 1))

In [ ]:
# ── 2.2 Fit degree 1, 3 and 7 to ten SIMULATED time-course measurements ──
rng = np.random.default_rng(SEED + 13)
y_obs = circadian(T_FIXED) + rng.normal(0, SIGMA, T_FIXED.size)        # the 10 training measurements
t_new = rng.uniform(0, 1, 2000)                                         # new measurements (test data)
y_new = circadian(t_new) + rng.normal(0, SIGMA, t_new.size)
tt = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, d in zip(axes, [1, 3, 7]):
    w = fit_poly(T_FIXED, y_obs, d)
    tr, te = np.mean((predict_poly(w, T_FIXED) - y_obs) ** 2), np.mean((predict_poly(w, t_new) - y_new) ** 2)
    ax.scatter(t_new[:40] * 24, y_new[:40], s=12, color="tab:brown", alpha=0.5, label="new data")
    ax.scatter(T_FIXED * 24, y_obs, s=25, color="tab:red", zorder=3, label="training data")
    ax.plot(tt * 24, predict_poly(w, tt), "k-", lw=1.5, label="model built")
    ax.plot(tt * 24, circadian(tt), color="tab:green", ls="--", lw=1, label="true f(t)")
    ax.set_title(f"degree {d}: train MSE {tr:.3f}, test MSE {te:.3f}", fontsize=9)
    ax.set_xlabel("time of day (h)"); ax.set_ylim(-2, 2)
axes[0].set_ylabel("log-expression (SIMULATED)"); axes[0].legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The three classic pictures, with numbers. The straight line (degree 1) is **rigid**: it
misses the daily rhythm, so both its training and its test error are high — **underfitting**. Degree 3 follows the
rhythm, and its training and test errors are both low. Degree 7 bends to chase individual noisy points: its training
error is the lowest of the three, but its test error is higher than degree 3's — **overfitting**. The training error
alone would have picked the worst-generalising curve of the two good-looking ones.

Let us look at all degrees from 0 to 9 at once (with 10 points, degree 9 passes through every point exactly).

In [ ]:
# ── 2.3 Training and test error for every degree (one data set) ──
degrees = np.arange(10)
train_err, test_err = [], []
for d in degrees:
    w = fit_poly(T_FIXED, y_obs, d)
    train_err.append(np.mean((predict_poly(w, T_FIXED) - y_obs) ** 2))
    test_err.append(np.mean((predict_poly(w, t_new) - y_new) ** 2))
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.plot(degrees, train_err, "o-", color="tab:blue", label="training error (10 points)")
ax.plot(degrees, test_err, "s-", color="tab:orange", label="test error (2,000 new points)")
ax.axhline(SIGMA ** 2, color="0.5", ls=":", label=r"noise variance $\sigma^2$")
ax.set_yscale("log"); ax.set_xlabel("polynomial degree (model complexity)"); ax.set_ylabel("MSE (log scale)")
ax.set_title("training error always falls; test error is U-shaped (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("degree:     ", " ".join(f"{d:6d}" for d in degrees))
print("train MSE:  ", " ".join(f"{e:6.3f}" for e in train_err))
print("test MSE:   ", " ".join(f"{e:6.2f}" for e in test_err))

🔍 **What just happened?** The training error can only go down as the degree increases (a degree-$d$ polynomial can do
anything a degree-$(d-1)$ one can), reaching zero at degree 9. The test error drops while the model gains the flexibility
it needs, is lowest at degree 3, and then climbs — slowly at first, steeply at degree 9. Note the dotted line: no model can have a test error below
the noise variance $\sigma^2=0.09$ on average — the measurements themselves scatter that much. In summary:
**high bias** ↔ rigid ↔ high training error; **low bias** ↔ flexible ↔ low training error; **high variance** ↔ not
robust to the training data (or noise) ↔ test error ≫ training error; **low variance** ↔ robust ↔ test error ≈ training
error. The simple model is robust (test error close to training error) but rigid; the complex model is flexible but
sensitive to noise. The best model balances the two.

⚠️ **Common misconception.** "Low training error means a good model." The training error of a flexible model is an
optimistic, biased estimate of its test error — it is the score of a student who wrote the exam questions.

✏️ **Your turn.** Change `SIGMA` to 0.1 in cell 2.1 and rerun cells 2.1–2.3. Does the best degree change? Now try 0.6. Explain why
noisier data favour simpler models.

---
## 3 · Bias and variance: what happens over many training sets

> 📍 **Where we are.** Section 2 fitted *one* data set. But the training set is itself random: repeat the experiment
> and you get different measurements and different fitted curves. Bias and variance describe that randomness.

High-level definitions: **bias** is a model's tendency to make *systematic* errors — the difference between its
*expected* prediction (averaged over many training sets) and the correct value we are trying to predict; **variance** is
how much its predictions *vary* from one training set to the next. Avoiding overfitting means a small error on new data,
which requires a trade-off between the two. The classic picture is a dartboard (Fortmann-Roe 2012): each dart is the
prediction of a model trained on a different data set, the bull's-eye is the truth. Low bias = darts centred on the
bull's-eye; low variance = darts tightly grouped.

In [ ]:
# ── 3.0 The dartboard picture of bias and variance (SIMULATED darts) ──
rng = np.random.default_rng(SEED + 30)
fig, axes = plt.subplots(2, 2, figsize=(6.4, 6.6))
boards = {(0, 0): ("low bias, low variance", (0.0, 0.0), 0.25), (0, 1): ("low bias, high variance", (0.0, 0.0), 0.85),
          (1, 0): ("high bias, low variance", (1.3, 0.9), 0.25), (1, 1): ("high bias, high variance", (1.3, 0.9), 0.85)}
for (i, j), (title, centre, spread) in boards.items():
    ax = axes[i, j]
    for radius, colour in zip([3.0, 2.25, 1.5, 0.75], ["#f3f3f3", "#cfe2f3", "#9fc5e8", "#e06666"]):
        ax.add_patch(plt.Circle((0, 0), radius, facecolor=colour, edgecolor="0.5"))
    darts = np.array(centre) + spread * rng.normal(size=(12, 2))      # 12 models from 12 training sets
    ax.scatter(darts[:, 0], darts[:, 1], s=16, color="k", zorder=3)
    ax.plot(*darts.mean(axis=0), "*", color="gold", ms=15, mec="k", zorder=4)
    ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(title, fontsize=9)
fig.suptitle("bull's-eye = truth · dot = one fitted model · ★ = average model (SIMULATED)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Bias is the distance from the gold star (the average model) to the bull's-eye; variance is the
spread of the dots around the star. Rigid models land in the bottom-left board (consistent but off-target), very
flexible models in the top-right (on target on average, but any single model may be far off). We now throw 200 actual
"darts" per model: polynomial fits to 200 independent simulated experiments.

In [ ]:
# ── 3.1 Two hundred repeats of the experiment: the fan of fitted curves ──
rng = np.random.default_rng(SEED + 3)
R = 200
Y_rep = circadian(T_FIXED)[:, None] + rng.normal(0, SIGMA, (T_FIXED.size, R))   # 10 × 200: one column per experiment
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, d in zip(axes, [1, 3, 7]):
    fits = predict_poly(fit_poly(T_FIXED, Y_rep, d), tt)                         # 300 × 200 fitted curves
    ax.plot(tt * 24, fits[:, :60], color="tab:blue", alpha=0.12, lw=0.8)
    ax.plot(tt * 24, fits.mean(axis=1), color="k", lw=2, label="average fit")
    ax.plot(tt * 24, circadian(tt), color="tab:green", ls="--", lw=1.5, label="true f(t)")
    bias2 = np.mean((fits.mean(axis=1) - circadian(tt)) ** 2)
    var = np.mean(fits.var(axis=1))
    ax.set_title(f"degree {d}: bias² {bias2:.3f}, variance {var:.3f}", fontsize=9)
    ax.set_xlabel("time of day (h)"); ax.set_ylim(-2, 2)
axes[0].set_ylabel("fitted log-expression"); axes[0].legend(fontsize=7)
fig.suptitle("60 of 200 fits to independent SIMULATED experiments", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Which model is the most rigid (biased)? Which varies most with the training data?* — now
measured. The **average** straight line (black) is far from the truth
whatever the data: large **bias**; but the individual lines barely move: small **variance**. The degree-7 curves are
right *on average* (their mean is almost on top of the green truth: tiny bias) but each individual fit swings wildly,
especially near the ends of the day: large variance. Degree 3 has both small. The titles quantify it, averaged over the
day: bias² is the squared distance between the black and green curves, variance the average spread of the blue fan.
A rigid model is unlikely to contain the true function (high bias); a flexible one probably contains it (low bias).

The same trade-off governs every model-selection choice of Section 1. Take **feature selection** in regression,
$f: X\to Y$ with $X = (X_1,\dots,X_{1000})$: a model using one feature has higher bias and lower variance; two features,
lower bias and higher variance; all 1,000 features, low bias but very high variance; a well-chosen subset
($X_1, X_3, X_{100}$, say) balances the two. *Which subset should we use?* Let us simulate a small GWAS
([Module 06](06_linear_regression_gwas.ipynb)): 60 individuals, 50 SNPs, of which 5 truly affect the trait.

In [ ]:
# ── 3.2 Feature subsets in a SIMULATED GWAS: one SNP, the true 5, 20, or all 50 ──
rng = np.random.default_rng(SEED + 32)
N_IND, J_SNP = 60, 50
G0, _, _ = simulate_genotypes(rng, 1000, J_SNP)
_, beta, causal = simulate_quantitative_trait(rng, G0, n_causal=5, h2=0.5)       # fixed true effects
noise_sd = np.sqrt(0.5)
others = np.setdiff1d(np.arange(J_SNP), causal)
subsets = {"1 SNP (strongest causal)": causal[[np.argmax(np.abs(beta[causal]))]],
           "the 5 causal SNPs": causal,
           "5 causal + 15 null SNPs": np.r_[causal, others[:15]],
           "all 50 SNPs": np.arange(J_SNP)}
G_test, _, _ = simulate_genotypes(rng, 500, J_SNP)                                 # fixed test individuals
f_test = G_test @ beta                                                             # their true genetic values
preds = {name: [] for name in subsets}
for r in range(300):                                                               # 300 independent studies
    G, _, _ = simulate_genotypes(rng, N_IND, J_SNP)
    y = G @ beta + rng.normal(0, noise_sd, N_IND)
    for name, cols in subsets.items():
        Xd = np.c_[np.ones(N_IND), G[:, cols]]
        w = np.linalg.lstsq(Xd, y, rcond=None)[0]
        preds[name].append(np.c_[np.ones(len(G_test)), G_test[:, cols]] @ w)
print(f"{'model':26s}  bias²   variance  noise   expected test MSE")
bv = {}
for name, P in preds.items():
    P = np.array(P)
    bv[name] = (np.mean((P.mean(0) - f_test) ** 2), np.mean(P.var(0)), noise_sd ** 2)
    print(f"{name:26s}  {bv[name][0]:.3f}   {bv[name][1]:.3f}    {bv[name][2]:.3f}   {sum(bv[name]):.3f}")

🔍 **What just happened?** Exactly the predicted pattern. One SNP: **high bias** (it ignores four real effects; bias² 0.48), low
variance (0.12). All 50 SNPs: essentially no bias but **huge variance** (5.4) — with 60 individuals and 51 coefficients every study's estimates
are dominated by noise. The right subset (the 5 causal SNPs) has the smallest expected test error; adding 15 null SNPs
leaves the bias at essentially zero but more than quadruples the variance (0.10 → 0.45). Every unnecessary parameter
is estimated from the same 60 individuals and adds its own noise to the predictions. Of course, in a real study we do not know which subset is right — that is the question "which
subset should we use?", answered by the tools of Sections 5–8 (and by the lasso,
[Module 07](07_ridge_lasso_sparse_regression.ipynb)).

**HMM states and Bayesian-network edges.** How many states should each hidden variable of an HMM have? A small number
of states: high bias, low variance; a large number: low bias, high variance. In a Bayesian network, a model with few edges
has high bias and low variance; one with many edges, low bias and high variance. More states or more edges mean more
parameters. Counting free parameters makes this concrete (we reuse these counts for BIC in Section 8):

In [ ]:
# ── 3.3 Counting free parameters: HMM states and Bayesian-network edges ──
M_SYMBOLS = 4                                                   # an HMM over DNA: 4 emission symbols
print("HMM with K states over 4 symbols: (K−1) initial + K(K−1) transition + K(4−1) emission parameters")
for K in range(1, 7):
    print(f"   K = {K}: {(K - 1) + K * (K - 1) + K * (M_SYMBOLS - 1):3d} free parameters")


def bn_parameters(edges, nodes, levels=3):
    """Free parameters of a discrete Bayesian network: sum over nodes of (levels − 1) · levels^(number of parents)."""
    n_par = {v: sum(1 for a, b in edges if b == v) for v in nodes}
    return sum((levels - 1) * levels ** n_par[v] for v in nodes)


complete = [(a, b) for i, a in enumerate(SACHS_PROTEINS) for b in SACHS_PROTEINS[i + 1:]]   # every possible edge
print(f"\n11 proteins, each low/medium/high:\n   no edges: {bn_parameters([], SACHS_PROTEINS)}"
      f" | Sachs consensus graph ({len(SACHS_CONSENSUS_EDGES)} edges): {bn_parameters(SACHS_CONSENSUS_EDGES, SACHS_PROTEINS)}"
      f" | complete DAG ({len(complete)} edges): {bn_parameters(complete, SACHS_PROTEINS):,}")

🔍 **What just happened?** Each extra HMM state adds a whole row of transition and emission probabilities, so the count
grows roughly quadratically in $K$. For Bayesian networks the growth is *exponential* in the number of parents: the
20-edge signalling network of [Module 17](17_bayesian_networks_inference_learning.ipynb) needs 230 free parameters
(22 with no edges at all), while a complete graph needs 177,146 — about a hundred times more than the 1,800 cells of the
discrete Sachs data.

✏️ **Your turn.** In cell 3.2, change `N_IND` from 60 to 200 individuals. Which model's expected test MSE improves the
most, and which term (bias² or variance) is responsible? Does the ranking of the four models change?

### 📘 Lecture notes — what model selection is, and the bias–variance trade-off

- **Parameter estimation vs model selection.** Learning usually means estimating parameters for a *known* model $M$
  (MLE, MAP or fully Bayesian for probabilistic models). Model selection asks how to learn $M$ itself: features for
  regression, $k$ for k-NN, tree size, $C$/kernel for SVMs, graph structure for Bayesian networks and GGMs, the number
  of clusters for k-means and mixtures, the number of states for HMMs, the architecture of a neural network.
- **What is a good model?** Three fits to the same data — rigid (line), moderate, very flexible (degree 7): training
  error high / middle / low; testing error high / low / high.
- **Bias:** a model's tendency to make systematic errors — the difference between its expected prediction (over
  training sets) and the correct value. **Variance:** how much its predictions vary with different training sets.
  Dartboard: bias = off-centre, variance = scatter.
- **Summary.** High bias = rigid, high training error; low bias = flexible, low training error. High variance = not
  robust to the training data or noise, test error ≫ training error; low variance = robust, test error ≈ training
  error. **Best model = bias–variance trade-off.**
- **Everywhere in the course.** One feature (high bias, low variance) vs all 1,000 features (low bias, high variance) vs
  a subset (balanced); few vs many HMM states; few vs many Bayesian-network edges.
- **Worked example (Section 3).** 60 individuals, 50 SNPs, 5 causal: one SNP has bias² 0.48 and variance 0.12; all 50
  SNPs have bias² ≈ 0 and variance 5.4; the 5 causal SNPs have the lowest expected test error.

> 🎓 **Exam tip.** Be ready to rank three pictured fits by training error, test error, bias and variance, and to explain why training error always decreases with complexity while test error is U-shaped.

---
## 4 · The bias–variance decomposition, derived and verified

> 📍 **Where we are.** Section 3 measured bias and variance by simulation. Now we look more closely at these two
> sources of error and prove that, for squared error, the expected test error is *exactly* their sum plus the noise. We
> derive it line by line, then give a second, alternative proof, and check every line numerically.

**Formulation.** Data are generated as $y=f(x)+\epsilon$ with $\mathbb E[\epsilon]=0$ and
$\mathrm{Var}(\epsilon)=\sigma^2$. From a training set $D$ we compute an estimate $\hat f(x;D)$. Its error at a new point
$x$ with a new measurement $y$ is $(y-\hat f(x;D))^2$. Averaged over training sets $D$ and over the new noise $\epsilon$:

$$\mathbb E_{D,\epsilon}\Big[\big(y-\hat f(x;D)\big)^2\Big]=\Big(\mathrm{Bias}_D\big[\hat f(x;D)\big]\Big)^2+\mathrm{Var}_D\big[\hat f(x;D)\big]+\sigma^2,$$

with $\mathrm{Bias}_D[\hat f(x;D)]=\mathbb E_D[\hat f(x;D)-f(x)]$.

*In words:* the expected squared error of a learning method at $x$ splits into three non-negative parts: how far its
*average* prediction is from the truth (bias, squared), how much its prediction wobbles from training set to training
set (variance), and the scatter of the new measurement itself (noise), which no method can remove.

🧮 **Math refresher — expectations.** $\mathbb E[Z]$ is the long-run average of a random quantity $Z$ over infinitely many
repetitions. Three rules do all the work: **linearity**, $\mathbb E[aZ+bW+c]=a\mathbb E[Z]+b\mathbb E[W]+c$ for constants
$a,b,c$; **variance**, $\mathrm{Var}(Z)=\mathbb E[(Z-\mathbb E Z)^2]=\mathbb E[Z^2]-(\mathbb E Z)^2$; and **independence**: if
$Z$ and $W$ are independent, $\mathbb E[ZW]=\mathbb E[Z]\,\mathbb E[W]$. A non-random quantity such as $f(x)$ is its own expectation.

### 4.1 The derivation

Abbreviate $f=f(x)$ and $\hat f=\hat f(x;D)$ and drop the subscripts on $\mathbb E$. Expanding the square and
using linearity:

$$\mathbb E\big[(y-\hat f)^2\big]=\mathbb E\big[y^2-2y\hat f+\hat f^2\big]=\mathbb E[y^2]-2\,\mathbb E[y\hat f]+\mathbb E[\hat f^2].$$

*In words:* the expected squared error is the sum of three simpler expectations, which we compute one at a time.

**First**, since $y=f+\epsilon$ and $f$ is a fixed number:
$$\mathbb E[y^2]=\mathbb E[(f+\epsilon)^2]=f^2+2f\,\mathbb E[\epsilon]+\mathbb E[\epsilon^2]=f^2+0+\sigma^2.$$
*In words:* a measurement's mean square is the true value squared plus the noise variance ($\mathbb E[\epsilon^2]=\sigma^2$
because $\mathbb E[\epsilon]=0$).

**Secondly**:
$$\mathbb E[y\hat f]=\mathbb E[(f+\epsilon)\hat f]=f\,\mathbb E[\hat f]+\mathbb E[\epsilon\hat f]=f\,\mathbb E[\hat f]+\mathbb E[\epsilon]\,\mathbb E[\hat f]=f\,\mathbb E[\hat f].$$
*In words:* the step $\mathbb E[\epsilon\hat f]=\mathbb E[\epsilon]\mathbb E[\hat f]$ needs the **new** measurement's noise to be
independent of the training data that produced $\hat f$ — true for a genuinely new test point, and exactly what fails if
the "test" point was used in training. Derivations often use this silently.

**Lastly**, from the variance rule: $\mathbb E[\hat f^2]=\mathrm{Var}(\hat f)+(\mathbb E[\hat f])^2.$

**Plugging in**:
$$\mathbb E\big[(y-\hat f)^2\big]=f^2+\sigma^2-2f\,\mathbb E[\hat f]+\mathrm{Var}(\hat f)+(\mathbb E[\hat f])^2
=\underbrace{\big(f-\mathbb E[\hat f]\big)^2}_{\text{Bias}^2}+\underbrace{\mathrm{Var}(\hat f)}_{\text{Variance}}+\sigma^2.$$
*In words:* $f^2-2f\mathbb E[\hat f]+(\mathbb E[\hat f])^2$ is a perfect square, $(f-\mathbb E[\hat f])^2$; what remains is the
variance of the estimate and the noise.

In [ ]:
# ── 4.2 Verify every line at one time point (t = 6 h, the expression peak) with 20,000 simulated experiments ──
rng = np.random.default_rng(SEED + 4)
t0, R_BIG = 0.25, 20_000
f0 = circadian(t0)
Y_many = circadian(T_FIXED)[:, None] + rng.normal(0, SIGMA, (T_FIXED.size, R_BIG))   # 20,000 training sets
y0 = f0 + rng.normal(0, SIGMA, R_BIG)                                                # a NEW measurement for each
for d in [1, 3, 7]:
    fhat = predict_poly(fit_poly(T_FIXED, Y_many, d), np.array([t0]))[0]            # 20,000 predictions at t0
    print(f"degree {d}:")
    print(f"   E[y²]   = {np.mean(y0 ** 2):.4f}   vs  f² + σ²            = {f0 ** 2 + SIGMA ** 2:.4f}")
    print(f"   E[y·f̂]  = {np.mean(y0 * fhat):.4f}   vs  f·E[f̂]             = {f0 * fhat.mean():.4f}")
    print(f"   E[f̂²]   = {np.mean(fhat ** 2):.4f}   vs  Var(f̂) + E[f̂]²      = {fhat.var() + fhat.mean() ** 2:.4f}")
    bias2, var = (f0 - fhat.mean()) ** 2, fhat.var()
    print(f"   E[(y−f̂)²] measured = {np.mean((y0 - fhat) ** 2):.4f}  |  bias² {bias2:.4f} + variance {var:.4f} "
          f"+ σ² {SIGMA ** 2:.4f} = {bias2 + var + SIGMA ** 2:.4f}")

🔍 **What just happened?** For each degree, each of the three intermediate identities holds to Monte Carlo precision
(the small discrepancies shrink as you increase `R_BIG`), and the directly measured expected error equals bias² +
variance + σ². At this time point the straight line's error is dominated by bias² (0.44 of 0.54), while degree 7's
reducible error is pure variance (bias² ≈ 0, variance 0.067); degree 3 has the smallest total (0.12). The decomposition is not an approximation: it is an algebraic identity about averages.

### 4.2 An alternative proof via the optimal predictor

This proof uses different letters: $y$ is the true output, $h(x)=\mathbb E[y\mid x]$ is the **optimal predictor** (the
true regression function, called $f$ above), and $f(x;D)$ is *our estimate* (called $\hat f$ above). For a fixed training
set $D$, average over a new test pair $(x,y)$ with density $p(x,y)$ and insert $\pm h(x)$:

$$\int\big(f(x;D)-y\big)^2p(x,y)\,dx\,dy=\int\big(f(x;D)-h(x)\big)^2p(x)\,dx+\int\big(h(x)-y\big)^2p(x,y)\,dx\,dy.$$

*In words:* the error splits into "distance of our predictor from the best possible predictor" plus "the irreducible
scatter of $y$ around its conditional mean". The cross term $2\int(f-h)(h-y)\,p(x,y)$ vanishes because, for each $x$,
the average of $h(x)-y$ over $y$ is zero — that is the definition of $h(x)=\mathbb E[y\mid x]$.

Then average the first term over training sets, inserting $\pm\mathbb E_D[f(x;D)]$:

$$\mathbb E_D\Big[\big(f(x;D)-h(x)\big)^2\Big]=\underbrace{\big(\mathbb E_D[f(x;D)]-h(x)\big)^2}_{\text{Bias}^2}+\underbrace{\mathbb E_D\Big[\big(f(x;D)-\mathbb E_D[f(x;D)]\big)^2\Big]}_{\text{Variance}}.$$

*In words:* the same trick again — write the expansion as $a+b+c$ with $a$ = variance, $b$ = the cross term, $c$ = bias²; the cross term $b$ is zero because $f(x;D)-\mathbb E_D[f(x;D)]$ averages
to zero over training sets. The two proofs agree; the second one shows that the noise term is really
"$\mathbb E[(y-\mathbb E[y\mid x])^2]$", which need not be a constant $\sigma^2$.

⚠️ **Notation trap.** This first step is sometimes written with $E_D[(f(x;D)-y)^2]$ on the left, but the integral on the
right is over the test pair $(x,y)$ for fixed $D$; the expectation over $D$ is taken in the next step. Keep track of *which*
randomness each $\mathbb E$ averages over.

In [ ]:
# ── 4.3 Bias², variance and noise for every degree, averaged over the day ──
rng = np.random.default_rng(SEED + 43)
t_grid = np.linspace(0, 1, 101)
Y_rep = circadian(T_FIXED)[:, None] + rng.normal(0, SIGMA, (T_FIXED.size, 4000))
rows = []
for d in degrees:
    W = fit_poly(T_FIXED, Y_rep, d)
    P = predict_poly(W, t_grid)                                         # 101 × 4000 predictions
    bias2 = np.mean((P.mean(axis=1) - circadian(t_grid)) ** 2)
    var = np.mean(P.var(axis=1))
    y_fresh = circadian(t_grid)[:, None] + rng.normal(0, SIGMA, P.shape)
    train = np.mean((predict_poly(W, T_FIXED) - Y_rep) ** 2)
    rows.append((d, train, bias2, var, bias2 + var + SIGMA ** 2, np.mean((y_fresh - P) ** 2)))
bvt = pd.DataFrame(rows, columns=["degree", "train MSE", "bias²", "variance", "bias²+var+σ²", "test MSE (direct)"])
print(bvt.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(bvt["degree"], bvt["bias²"], "o-", color="tab:red", label="bias²")
ax.plot(bvt["degree"], bvt["variance"], "o-", color="tab:cyan", label="variance")
ax.plot(bvt["degree"], bvt["test MSE (direct)"], "s-", color="k", label="expected test error")
ax.plot(bvt["degree"], bvt["train MSE"], "--", color="tab:blue", label="expected training error")
ax.axhline(SIGMA ** 2, color="0.5", ls=":", label=r"noise $\sigma^2$")
ax.set_ylim(0, 0.7); ax.set_xlabel("polynomial degree (model complexity)"); ax.set_ylabel("mean squared error")
ax.set_title("measured: bias² falls, variance rises (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** This is the famous figure of the bias–variance trade-off (underfitting on the left,
overfitting on the right), computed rather than drawn. Bias² collapses
once the polynomial can represent a sine wave (degree 3, and essentially zero from degree 5), while the variance grows
steadily and then explodes at degree 9, where the curve must pass through every noisy point. Their sum plus $\sigma^2$
matches the directly simulated test error in every row. The expected **training** error keeps falling and is *below* $\sigma^2$ as soon
as the model has a few parameters (for least squares its expectation is roughly $\sigma^2(1-p/n)$ plus bias, with $p$
coefficients and $n=10$ points) — the fit absorbs part of the noise. The best degree here is 3, where the total is smallest: the
**bias–variance trade-off**. (Degree 5 has almost no bias but more variance; degrees 4–6 are close.)

✏️ **Your turn.** Increase the number of time points from 10 to 30 (`T_FIXED = np.linspace(0, 1, 30)`) and rerun 4.3.
Which term changes, and how does the best degree move? Why does more data let you afford a more complex model?

### 📘 Lecture notes — the bias–variance decomposition of the mean squared error

- **Set-up.** $y = f(x)+\epsilon$, $\mathbb E[\epsilon]=0$, $\mathrm{Var}(\epsilon)=\sigma^2$; from training data $D$ we
  estimate $\hat f(x;D)$.
- **Result.**
  $$\mathbb E_{D,\epsilon}\big[(y-\hat f)^2\big] = \underbrace{\big(f-\mathbb E[\hat f]\big)^2}_{\text{Bias}^2} + \underbrace{\mathrm{Var}(\hat f)}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{noise (irreducible)}} .$$
  *In words:* expected test error = systematic error² + instability + scatter of the new measurement.
- **Proof 1 (three expectations).** $\mathbb E[(y-\hat f)^2] = \mathbb E[y^2] - 2\mathbb E[y\hat f] + \mathbb E[\hat f^2]$ with
  $\mathbb E[y^2] = f^2+\sigma^2$, $\mathbb E[y\hat f] = f\,\mathbb E[\hat f]$ (new noise independent of $D$), $\mathbb E[\hat f^2] =
  \mathrm{Var}(\hat f) + \mathbb E[\hat f]^2$; then complete the square.
- **Proof 2 (optimal predictor).** With $h(x) = \mathbb E[y\mid x]$, insert $\pm h(x)$: the error splits into
  $\int(f(x;D)-h(x))^2p(x)\,dx$ + $\int(h(x)-y)^2p(x,y)\,dx\,dy$ (the cross term vanishes because $h$ is the conditional mean);
  then insert $\pm\mathbb E_D[f(x;D)]$ in the first term: $a$ (variance) + $b$ (cross term, zero) + $c$ (bias²).
- **Shape.** As complexity grows (from $y = w_0 + wx + \epsilon$ to a degree-7 polynomial), bias² falls, variance rises,
  and their sum plus $\sigma^2$ is U-shaped.
- **Worked example (Section 4.2, $t$ = 6 h).** Straight line: bias² 0.44 of a total 0.54; degree 7: bias² ≈ 0, variance
  0.067; degree 3: smallest total, 0.12.

> 🎓 **Exam tip.** The derivation is a classic exam question: expand the square, compute the three expectations, complete the square. State where you use $\mathbb E[\epsilon]=0$ and the independence of the test noise from $D$.

---
## 5 · Cross-validation from scratch

> 📍 **Where we are.** In simulations we could compute test errors exactly; with real data we cannot repeat the
> experiment. *How can we find the best model and combat overfitting?* There are three weapons: **cross-validation**,
> **regularization** and the **Bayesian information criterion**. This section builds the first.

**Hold-out validation.** Split the *training data* into a training set and a **validation set**. For
each candidate model, estimate parameters on the training set and score it (likelihood or error) on the validation set;
pick the best score. The **test data** stay locked away until the very end and are used once, to report the
performance of the chosen model. *You should not use test data during training with CV — otherwise you would be
overfitting* to the test set.

**k-fold cross-validation.** Split the training data into $k$ subsets (folds). For each fold, train on the
other $k-1$ folds and compute the score on the held-out fold; average the $k$ scores:

$$\mathrm{CV}_k=\frac1N\sum_{j=1}^{k}\ \sum_{i\in\text{fold }j}\Big(y_i-\hat f^{(-j)}(x_i)\Big)^2.$$

*In words:* every sample is predicted exactly once, by a model that never saw it ($\hat f^{(-j)}$ is trained without
fold $j$); the CV error is the average of these honest prediction errors. **Leave-one-out CV** (LOOCV) is
the special case $k=N$: each fold holds a single sample.

In [ ]:
# ── 5.1 k-fold and leave-one-out cross-validation, written from scratch ──
def kfold_indices(n, k, rng):
    """Randomly split the sample indices 0 … n−1 into k folds of (almost) equal size."""
    return np.array_split(rng.permutation(n), k)


def cv_mse_poly(t, y, degree, k, rng):
    """k-fold CV estimate of the test MSE of a degree-`degree` polynomial (k = len(y) gives LOOCV)."""
    squared_errors = np.empty(len(y))
    for fold in kfold_indices(len(y), k, rng):
        train = np.setdiff1d(np.arange(len(y)), fold)          # everything except this fold
        w = fit_poly(t[train], y[train], degree)               # parameter estimation on the training part
        squared_errors[fold] = (predict_poly(w, t[fold]) - y[fold]) ** 2   # score on the held-out part
    return squared_errors.mean()


rng = np.random.default_rng(SEED + 5)
folds = kfold_indices(10, 3, rng)
print("3-fold split of 10 samples:", [f.tolist() for f in folds])
print("LOOCV on 10 samples = 10 folds of size", {len(f) for f in kfold_indices(10, 10, rng)})

**A LOOCV example.** The lecture compares two models on nine points by LOOCV: Model 1 (a straight line) has
$\mathrm{MSE}_{\text{LOOCV}}=2.12$ and Model 2 (a quadratic) $0.962$, so Model 2 wins. We make our own nine SIMULATED
points with a similar arch shape; our numbers will differ, the logic will not.

In [ ]:
# ── 5.2 LOOCV for a line vs a quadratic on nine points ──
rng = np.random.default_rng(SEED + 58)
x9 = np.linspace(0.3, 5.7, 9)                                     # nine evenly spaced inputs
y9 = 4 - 0.4 * (x9 - 3) ** 2 + rng.normal(0, 0.6, 9)              # an arch plus noise (SIMULATED)
xx = np.linspace(-0.3, 6.3, 200)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, (d, name) in zip(axes, [(1, "Model 1: line"), (2, "Model 2: quadratic")]):
    loo = cv_mse_poly(x9, y9, d, k=9, rng=rng)
    for i in range(9):                                            # the nine leave-one-out fits
        keep = np.arange(9) != i
        w = fit_poly(x9[keep], y9[keep], d)
        ax.plot(xx, predict_poly(w, xx), color="tab:green", alpha=0.35, lw=1)
        ax.plot([x9[i], x9[i]], [y9[i], predict_poly(w, x9[i:i + 1])[0]], color="tab:red", lw=1.2)
    ax.scatter(x9, y9, color="k", zorder=3, s=20)
    ax.set_title(f"{name}: MSE_LOOCV = {loo:.2f}", fontsize=10); ax.set_xlabel("x")
axes[0].set_ylabel("y (SIMULATED)")
fig.suptitle("green: the 9 fits, each without one point; red: the error on the left-out point", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each green curve was fitted without one of the nine points; the red segment is its error on
that point. The line cannot follow the arch, so its left-out errors are large, especially at the ends; the quadratic's
are small. As in the lecture's example, LOOCV prefers Model 2 — here by an even larger factor (2.32 vs 0.45, about 5×, against
2.12 vs 0.962), because our arch is more pronounced. Note that LOOCV needs no extra data: all nine points are used both for fitting (eight at a time) and for
scoring (one at a time).

Now we use CV for what it is for: **choosing a model**. We give ourselves a realistic budget of 30 measurements at random
times and choose the polynomial degree by 10-fold CV, comparing with the true test error (which only a simulation can give).

In [ ]:
# ── 5.3 Choosing the polynomial degree by 10-fold CV ──
rng = np.random.default_rng(SEED + 65)
t30 = rng.uniform(0, 1, 30)
y30 = circadian(t30) + rng.normal(0, SIGMA, 30)
cv_curve = [cv_mse_poly(t30, y30, d, k=10, rng=np.random.default_rng(SEED)) for d in degrees]
true_curve = [np.mean((predict_poly(fit_poly(t30, y30, d), t_new) - y_new) ** 2) for d in degrees]
train_curve = [np.mean((predict_poly(fit_poly(t30, y30, d), t30) - y30) ** 2) for d in degrees]
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.plot(degrees, train_curve, "o--", color="tab:blue", label="training error")
ax.plot(degrees, cv_curve, "o-", color="tab:purple", label="10-fold CV estimate")
ax.plot(degrees, true_curve, "s-", color="tab:orange", label="true test error (2,000 new points)")
ax.set_yscale("log"); ax.set_xlabel("polynomial degree"); ax.set_ylabel("MSE (log scale)")
ax.set_title("CV tracks the test error using only the 30 training points"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
d_cv, d_true = int(np.argmin(cv_curve)), int(np.argmin(true_curve))
print(f"degree chosen by 10-fold CV: {d_cv} (true test error {true_curve[d_cv]:.3f}) | "
      f"best degree on the true test error: {d_true} ({true_curve[d_true]:.3f})")
holdout_choice = []                                   # for comparison: ONE hold-out split, repeated
for s in range(10):
    perm = np.random.default_rng(100 + s).permutation(30)
    val, trn = perm[:9], perm[9:]                     # 21 points for training, 9 for validation
    errs = [np.mean((predict_poly(fit_poly(t30[trn], y30[trn], d), t30[val]) - y30[val]) ** 2) for d in degrees]
    holdout_choice.append(int(np.argmin(errs)))
print("degree chosen by a single 21/9 hold-out split, for 10 different random splits:", holdout_choice)

🔍 **What just happened?** Using nothing but the 30 training measurements, the CV curve reproduces the U shape of the
true test error, and here its minimum lands exactly on the true best degree (3). On another data set CV may well pick
4 or 5, whose true errors are almost the same — Section 8.1 shows the choices over 200 data sets. The last line shows
why we prefer k-fold CV to a single hold-out split: with only 9 validation points, the chosen degree depends on *which*
9 points were held out (here anything from 3 to 9 over ten random splits), whereas k-fold CV uses every point for
validation exactly once and averages over the folds. The training
error, by contrast, keeps falling and would always choose the most complex model. CV estimates are themselves noisy,
though: with a different random fold assignment the curve wiggles a little, and for high degrees (where a fold may leave
an end of the day uncovered) it can be erratic.

### 5.4 How many folds?

The practical rule: a large $k$ gives a more accurate assessment but costs $k$ model fits; small $k$ is cheaper but less accurate;
$k=5$ or $10$ are common; with small data sets, use large $k$ (LOOCV). Let us measure "accurate". We draw 300 independent
data sets of 30 measurements, compute the CV estimate of a cubic's test error with $k=2,5,10,30$, and compare it with the
cubic's true test error.

In [ ]:
# ── 5.4 The CV estimate itself has bias and variance: k = 2, 5, 10, 30 (LOOCV) ──
rng = np.random.default_rng(SEED + 54)
K_LIST, N30 = [2, 5, 10, 30], 30
est, truth = {k: [] for k in K_LIST}, []
for r in range(300):
    t_r = rng.uniform(0, 1, N30)
    y_r = circadian(t_r) + rng.normal(0, SIGMA, N30)
    truth.append(np.mean((predict_poly(fit_poly(t_r, y_r, 3), t_new) - y_new) ** 2))   # error of THIS fitted cubic
    for k in K_LIST:
        est[k].append(cv_mse_poly(t_r, y_r, 3, k, rng))
truth = np.array(truth)
print(f"true test error of a cubic fitted to 30 points: mean {truth.mean():.4f} (sd over data sets {truth.std():.4f})")
print(" k   model fits   mean CV estimate   median   sd of estimate   corr(estimate, this fit's true error)")
for k in K_LIST:
    e = np.array(est[k])
    print(f"{k:2d}   {k:6d}         {e.mean():.4f}          {np.median(e):.4f}     {e.std():.4f}            {np.corrcoef(e, truth)[0, 1]:+.2f}")
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.boxplot([np.array(est[k]) for k in K_LIST], showfliers=False)
ax.set_xticks(range(1, 5), [f"k = {k}" + (" (LOOCV)" if k == 30 else "") for k in K_LIST])
ax.axhline(truth.mean(), color="tab:orange", ls="--", label="mean true test error")
ax.set_ylabel("CV estimate of test MSE"); ax.set_title("300 SIMULATED data sets of 30 points (outliers hidden)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Three lessons, all visible in the table.

1. **Bias of the estimate.** With $k=2$ each model is trained on only 15 points, so it is worse than the model trained on
   all 30 — the 2-fold estimate is **pessimistic** (its mean is clearly above the true error, and occasionally a cubic
   fitted to 15 points extrapolates wildly, which inflates the mean far beyond the median). With $k=5,10,30$ the training
   sets are close to 30 points and the estimates are nearly unbiased.
2. **Variance of the estimate and cost.** For $k\ge5$ the spread of the estimate across data sets is similar here, while
   the cost grows linearly with $k$ (30 fits for LOOCV vs 5 for 5-fold). That is why $k=5$ or $10$ is the usual
   compromise; LOOCV is attractive when data are scarce or fits are cheap.
3. **What CV estimates.** The correlation between a data set's CV estimate and *that* fitted model's true error is about
   zero: CV tells you how well the *procedure* "fit a cubic to 30 points" works on average, not how good this particular
   fit is (Hastie *et al.* 2009, §7.12). The data-set-to-data-set spread of the estimate (sd ≈ 0.03–0.04) is also larger
   than the spread of the true error itself: a single CV number is noisy.

⚠️ **The number of folds is not the main danger.** The next experiment shows a mistake that no choice of $k$ can fix.

### 5.5 Correlated samples: independence matters more than k

CV assumes that the held-out samples are *independent* of the training samples, like genuinely new patients. Biological
data often violate this: several biopsies per patient, technical replicates, cells from the same mouse, related
individuals, samples processed in the same batch. Here 30 patients each contribute 4 biopsies; the 50 "expression"
features carry a strong patient-specific signature but **no information about the disease label** (labels are assigned at
random to patients). An honest classifier must be at chance (50 %).

In [ ]:
# ── 5.5 Record-wise vs patient-wise CV on SIMULATED repeated biopsies ──
def knn_predict(X_train, y_train, X_test, k=1):
    """k-nearest-neighbour majority vote (Euclidean distance; ties go to class 1)."""
    d2 = ((X_test[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
    nearest = np.argsort(d2, axis=1, kind="stable")[:, :k]
    return (y_train[nearest].mean(axis=1) >= 0.5).astype(int)


def cv_accuracy(X, y, folds, k=1):
    """Accuracy of k-NN when each fold (an index array) is predicted from all the other samples."""
    correct = 0
    for fold in folds:
        train = np.setdiff1d(np.arange(len(y)), fold)
        correct += np.sum(knn_predict(X[train], y[train], X[fold], k) == y[fold])
    return correct / len(y)


rng = np.random.default_rng(SEED + 55)
N_PAT, N_BIOPSY, J_FEAT = 30, 4, 50
res = []
for r in range(20):
    signature = rng.normal(0, 1, (N_PAT, J_FEAT))                         # each patient's own expression profile
    label = rng.permutation(np.r_[np.zeros(15, int), np.ones(15, int)])   # disease status: unrelated to profile
    X = np.repeat(signature, N_BIOPSY, axis=0) + 0.5 * rng.normal(0, 1, (N_PAT * N_BIOPSY, J_FEAT))
    y, patient = np.repeat(label, N_BIOPSY), np.repeat(np.arange(N_PAT), N_BIOPSY)
    rec5 = cv_accuracy(X, y, kfold_indices(len(y), 5, rng))
    loo = cv_accuracy(X, y, kfold_indices(len(y), len(y), rng))
    pat5 = cv_accuracy(X, y, [np.flatnonzero(np.isin(patient, g)) for g in kfold_indices(N_PAT, 5, rng)])
    res.append((rec5, loo, pat5))
res = np.array(res)
print("mean accuracy over 20 SIMULATED studies (truth: 0.50, the labels carry no signal)")
print(f"   record-wise 5-fold CV : {res[:, 0].mean():.3f}")
print(f"   record-wise LOOCV     : {res[:, 1].mean():.3f}")
print(f"   patient-wise 5-fold CV: {res[:, 2].mean():.3f}  (all biopsies of a patient in the same fold)")

🔍 **What just happened?** When biopsies are assigned to folds individually ("record-wise"), almost every held-out
biopsy has a sibling biopsy of the same patient in the training set; the 1-nearest-neighbour classifier simply
recognises the patient and copies the label, giving near-perfect "accuracy" on labels that are pure noise. Using more
folds (LOOCV) makes it *worse*, not better. Putting all biopsies of a patient in the same fold ("grouped" or
"patient-wise" CV) restores the honest answer, about 50 %. **The unit of cross-validation must be the unit you want to
generalise to** — new patients, new mice, new batches, new families — and this matters far more than the choice of $k$.

### 5.6 Cross-validation in action: tuning an SVM (the correction to "SVM: NA")

Now a larger data set and the model-selection choices that "SVM: NA" hides. The Wisconsin diagnostic breast-cancer
study (Street *et al.* 1993) measured 30 nuclear-shape features on images of fine-needle aspirates of 569 breast tumours
(63 % benign). We use `simulate_tumor_features()`, a simulated look-alike with the same 30 features, the same class
sizes and strongly correlated features (radius, perimeter and area are tied by geometry); its label is **1 = malignant,
0 = benign** (careful: scikit-learn's copy of the real data codes them the other way round). The tumours are classified with an RBF-kernel
SVM ([Module 11](11_support_vector_machines.ipynb)) for a grid of $C$ (penalty on margin violations) and $\gamma$ (kernel
width). Features are standardised **inside** each fold, using the training part's means and SDs only.

In [ ]:
# ── 5.6 5-fold CV accuracy of an RBF SVM over C × gamma (SIMULATED breast-tumour features) ──
bc = simulate_tumor_features()
X_bc, y_bc = bc["X"].to_numpy(), bc["y"]                           # y: 1 = malignant, 0 = benign
print(bc["source"], "|", X_bc.shape, "| malignant:", y_bc.sum(), "benign:", (y_bc == 0).sum())
folds_bc = kfold_indices(len(y_bc), 5, np.random.default_rng(SEED + 56))


def svm_cv_accuracy(C, gamma, scale=True):
    """5-fold CV accuracy of SVC(C, gamma); standardisation is fitted on the training folds only."""
    correct = 0
    for fold in folds_bc:
        train = np.setdiff1d(np.arange(len(y_bc)), fold)
        Xtr, mu, sd = standardize(X_bc[train]) if scale else (X_bc[train], 0.0, 1.0)
        Xte = standardize(X_bc[fold], mu, sd)[0] if scale else X_bc[fold]
        correct += np.sum(SVC(C=C, gamma=gamma).fit(Xtr, y_bc[train]).predict(Xte) == y_bc[fold])
    return correct / len(y_bc)


Cs, gammas = 10.0 ** np.arange(-2, 4), 10.0 ** np.arange(-4, 2)
acc_grid = np.array([[svm_cv_accuracy(C, g) for g in gammas] for C in Cs])
fig, ax = plt.subplots(figsize=(5.5, 3.8))
im = ax.imshow(acc_grid, cmap="viridis", vmin=0.6, vmax=1.0)
for i in range(len(Cs)):
    for j in range(len(gammas)):
        ax.text(j, i, f"{acc_grid[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="w" if acc_grid[i, j] < 0.8 else "k")
ax.set_xticks(range(len(gammas)), [f"{g:g}" for g in gammas]); ax.set_yticks(range(len(Cs)), [f"{c:g}" for c in Cs])
ax.set_xlabel("RBF kernel γ"); ax.set_ylabel("C"); fig.colorbar(im, label="5-fold CV accuracy")
ax.set_title("RBF SVM, SIMULATED breast-tumour data: C and γ matter")
plt.tight_layout(); plt.show()
print(f"accuracy range over the grid: {acc_grid.min():.3f} – {acc_grid.max():.3f} (always predicting 'benign': {1 - y_bc.mean():.3f})")
print(f"default C = 1, gamma = 'scale' WITHOUT standardising the features: {svm_cv_accuracy(1.0, 'scale', scale=False):.3f}")
for col in ["worst area", "mean smoothness"]:                      # features on very different scales
    print(f"range of '{col}': {bc['X'][col].min():.2f} – {bc['X'][col].max():.2f}")

🔍 **What just happened?** Depending on $C$ and $\gamma$ the same SVM is anything from useless (0.63: it predicts "benign"
for everyone, when $C$ is tiny or $\gamma$ so large that every tumour only resembles itself) to very good (0.95).
Skipping feature standardisation — another modelling choice — gives 0.90 with scikit-learn's defaults ($C=1$,
$\gamma$ = "scale"), several percentage points below the standardised grid's best, because features on large scales
dominate the kernel distance ("worst area" runs from about 140 to 2,300, "mean smoothness" from 0.06 to 0.14). So SVMs most definitely have model selection; "SVM: NA"
should read "$C$, kernel and kernel parameters (e.g. $\gamma$), class weights, preprocessing".

⚠️ **The best cell of this grid is an optimistic estimate.** We looked at 36 CV accuracies and picked the largest; some
of its advantage is luck. To report an honest accuracy for "SVM with $C$ and $\gamma$ tuned by CV", the tuning must
itself be inside an outer cross-validation loop — **nested CV**, Section 10.

✏️ **Your turn.** (a) Replace the RBF kernel by a polynomial kernel (`SVC(C=C, kernel="poly", degree=d)`) and make a
grid over $C$ and the degree $d\in\{1,2,3,4\}$. (b) In 5.5, weaken the patient signature (multiply `signature` by 0.2):
how much does record-wise CV still overestimate the accuracy?

### 📘 Lecture notes — cross-validation

- **Three weapons against overfitting:** cross-validation, regularization, the Bayesian information criterion.
- **Hold-out validation.** Split the *training* data into a training set and a validation set. For each model:
  estimate parameters on the training set, score the model on the validation set (likelihood or error); select the
  best score; deploy on the test data. **Never use test data during training or model selection** — otherwise you
  overfit to the test set.
- **k-fold CV.** Split the training data into $k$ subsets; each serves once as validation set while the rest train;
  pick the model with the best *average* score:
  $$\mathrm{CV}_k=\frac1N\sum_{j=1}^{k}\sum_{i\in\text{fold }j}\big(y_i-\hat f^{(-j)}(x_i)\big)^2 .$$
  *In words:* every sample is predicted once by a model that never saw it.
- **LOOCV** = $N$-fold CV: one sample per validation fold. Worked example from the lecture: on nine points a straight
  line has $\mathrm{MSE}_{\text{LOOCV}} = 2.12$ and a quadratic $0.962$ → choose the quadratic.
- **Choosing $k$.** Large $k$: more accurate assessment, but $k$ trainings. Small $k$: cheaper, less accurate (and
  pessimistic). Common: $k$ = 5 or 10; with small training sets use large $k$ (LOOCV) to train on as many samples as
  possible.
- **Beyond the basics.** Folds must respect the biological unit (all biopsies of a patient in one fold); a minimum over a
  large grid of CV scores is optimistic.

> 🎓 **Exam tip.** Know the algorithm of k-fold CV precisely (who is trained on what, what is averaged), why 2-fold CV is pessimistic, why LOOCV costs $N$ fits, and why replicate samples of one patient must stay in the same fold.

---
## 6 · The cardinal rule and selection bias

> 📍 **Where we are.** The cardinal rule of Section 5 — never let the test data influence training — has a subtle version that has
> tripped up many published genomics studies. With thousands of genes and dozens of patients, *selecting the genes* is
> part of training.

**The wrong procedure (selection outside the CV loop).** (1) Rank all 3,571 genes by how well they separate the
classes, using **all** samples. (2) Keep the top $m$. (3) Cross-validate a classifier on those $m$ genes.

**The right procedure (selection inside).** Inside every fold: rank the genes using the training part only, keep the top
$m$, train the classifier, predict the held-out part.

In step (1) of the wrong procedure the held-out samples have already "voted" for the genes that make them look
separable. Ambroise & McLachlan (2002) called this **selection bias**. The cleanest demonstration uses data in which
there is nothing to find: pure noise.

🧮 **Gene ranking by a t-statistic.** For gene $j$ with class means $\bar x_{1j},\bar x_{0j}$, variances $s^2_{1j},s^2_{0j}$
and class sizes $n_1,n_0$:
$$t_j=\frac{\bar x_{1j}-\bar x_{0j}}{\sqrt{s^2_{1j}/n_1+s^2_{0j}/n_0}}.$$
*In words:* the difference of the class means measured in units of its own standard error (Welch's two-sample
t-statistic); a large $|t_j|$ means the gene separates the classes well *in this sample*.

In [ ]:
# ── 6.1 Gene ranking, a 3-NN classifier, and CV with selection outside or inside the folds ──
def t_scores(X, y):
    """Welch two-sample t-statistic of every column (gene) of X between classes y = 1 and y = 0."""
    a, b = X[y == 0], X[y == 1]
    se = np.sqrt(a.var(axis=0, ddof=1) / len(a) + b.var(axis=0, ddof=1) / len(b))
    return (b.mean(axis=0) - a.mean(axis=0)) / (se + 1e-12)


def top_genes(X, y, m):
    """Indices of the m genes with the largest |t| (ranked on the samples given)."""
    return np.argsort(-np.abs(t_scores(X, y)), kind="stable")[:m]


def cv_error_selection(X, y, m, folds, inside=True, k=3):
    """CV error of k-NN on the top-m genes; genes ranked inside each training fold, or once on ALL samples."""
    genes_all = top_genes(X, y, m)
    wrong = 0
    for fold in folds:
        train = np.setdiff1d(np.arange(len(y)), fold)
        genes = top_genes(X[train], y[train], m) if inside else genes_all
        Ztr, mu, sd = standardize(X[train][:, genes])
        wrong += np.sum(knn_predict(Ztr, y[train], standardize(X[fold][:, genes], mu, sd)[0], k) != y[fold])
    return wrong / len(y)


rng = np.random.default_rng(SEED + 6)
noise_res = []
for r in range(20):                                             # 20 data sets of PURE NOISE, Golub-sized
    X_noise = rng.normal(size=(72, 3571))
    y_noise = rng.permutation(np.r_[np.zeros(36, int), np.ones(36, int)])
    f10 = kfold_indices(72, 10, rng)
    noise_res.append((cv_error_selection(X_noise, y_noise, 10, f10, inside=False),
                      cv_error_selection(X_noise, y_noise, 10, f10, inside=True)))
noise_res = 1 - np.array(noise_res)                              # convert to accuracies
print("PURE NOISE, 72 samples × 3,571 genes, top 10 genes, 3-NN, 10-fold CV (honest accuracy = 0.50):")
print(f"   selection OUTSIDE the folds: mean CV accuracy {noise_res[:, 0].mean():.3f} (range {noise_res[:, 0].min():.2f}–{noise_res[:, 0].max():.2f})")
print(f"   selection INSIDE the folds : mean CV accuracy {noise_res[:, 1].mean():.3f} (range {noise_res[:, 1].min():.2f}–{noise_res[:, 1].max():.2f})")

🔍 **What just happened?** On data that contain no signal at all, the wrong procedure reports a cross-validated accuracy
of around 80 % — a publishable-looking "diagnostic signature" made of noise. Among 3,571 random genes, some separate
these particular 72 random labels well *by chance*, and choosing them with all samples bakes that chance into every
fold. Selection inside the folds gives the honest answer: about 50 %. Now the real leukaemia data. Golub's ALL/AML
signal is strong, so we also use a **permuted-label** version — real gene expression, with the real correlations
between genes, but labels shuffled so that no gene can truly predict them.

In [ ]:
# ── 6.2 The same comparison on Golub-like leukaemia data (SIMULATED): true labels and shuffled labels ──
golub = simulate_leukemia_expression()
Xg, yg = golub["X"], golub["y"]
print("source:", golub["source"], "| samples:", Xg.shape[0], "| genes:", Xg.shape[1], "| AML:", yg.sum(), "ALL:", (yg == 0).sum())
rng = np.random.default_rng(SEED + 62)
perm_res = []
for r in range(20):
    yp = rng.permutation(yg)                                    # shuffled labels: no real signal
    f10 = kfold_indices(72, 10, rng)
    perm_res.append((cv_error_selection(Xg, yp, 10, f10, inside=False), cv_error_selection(Xg, yp, 10, f10, inside=True)))
perm_res = 1 - np.array(perm_res)
print(f"SHUFFLED labels (honest accuracy ≈ 0.65 by always predicting ALL, or worse):")
print(f"   outside: {perm_res[:, 0].mean():.3f}   inside: {perm_res[:, 1].mean():.3f}")
print("TRUE labels, 10-fold CV accuracy of 3-NN:")
f10 = kfold_indices(72, 10, np.random.default_rng(SEED + 63))
for m in [10, 50, 200]:
    print(f"   top {m:3d} genes: outside {1 - cv_error_selection(Xg, yg, m, f10, inside=False):.3f} | "
          f"inside {1 - cv_error_selection(Xg, yg, m, f10, inside=True):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, resm, title in [(axes[0], noise_res, "pure noise (SIMULATED)"), (axes[1], perm_res, f"Golub, shuffled labels\n({golub['source']})")]:
    ax.hist(resm[:, 0], bins=np.linspace(0.2, 1, 17), alpha=0.7, color="tab:red", label="selection outside CV")
    ax.hist(resm[:, 1], bins=np.linspace(0.2, 1, 17), alpha=0.7, color="tab:blue", label="selection inside CV")
    ax.set_title(title, fontsize=9); ax.set_xlabel("10-fold CV accuracy (20 repeats)")
axes[0].axvline(0.5, color="k", ls=":"); axes[1].axvline(47 / 72, color="k", ls=":")
axes[0].set_ylabel("number of repeats"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With shuffled labels, the wrong procedure still claims about 82 % accuracy, far above the
65 % one gets by always predicting the majority class (ALL), whereas the honest procedure gives 54 % — *below* that
baseline (k-NN on genes that carry no signal does worse than simply guessing "ALL"). With the **true** labels both
procedures give near-perfect accuracies — the simulated ALL/AML difference is large, as in Golub's data — and the outside
version is equal or slightly higher (1.000 vs 0.986 with 10 genes). That is the treacherous part: on real data you
cannot tell from the number alone how much of it is selection bias. Only the correct procedure is trustworthy.

🧪 **Ground truth.** Simulated data tell us which genes really differ between ALL and AML (`true_effect`). Which genes
does the ranking pick, with true and with shuffled labels?

In [ ]:
# ── 6.3 🧪 Are the selected genes truly differentially expressed? ──
de = np.abs(golub["true_effect"]) > 0                            # truly different between ALL and AML
print(f"truly DE genes: {de.sum()} of {len(de)} ({de.mean():.0%}); strong named markers: {np.sum(np.abs(golub['true_effect']) >= 0.5)}")
top_true = top_genes(Xg, yg, 20)
print("top 20 genes, TRUE labels    :", ", ".join(golub["genes"][top_true]))
print(f"   truly DE: {de[top_true].sum()} of 20")
frac_shuffled = [de[top_genes(Xg, np.random.default_rng(SEED + s).permutation(yg), 20)].mean() for s in range(20)]
print(f"top 20 genes, SHUFFLED labels: on average {np.mean(frac_shuffled):.0%} truly DE (the base rate is {de.mean():.0%})")
# caveat: true_effect is 0 for the B-/T-lineage marker genes, yet B-lineage markers still differ ALL vs AML (≈ 0.8 log10)
b_markers = np.isin(golub["genes"], [g for g, (c, _) in LEUKEMIA_MARKERS.items() if c == "B-ALL"])
print(f"B-lineage markers (true_effect = 0): mean ALL − AML difference "
      f"{(Xg[yg == 0][:, b_markers].mean(0) - Xg[yg == 1][:, b_markers].mean(0)).mean():.2f} log10")

🔍 **What just happened?** With the true labels, the 20 highest-ranked genes are all truly differentially expressed, and
most of them are the named markers of the simulation (the genes that distinguished the leukaemias in Golub's study). With
shuffled labels, the "top genes" are truly DE only at about the base rate — whatever the ranking picks is chance.
(One caveat: `true_effect` counts only the ALL-vs-AML programme; the B-lineage marker genes have `true_effect` = 0 but
still differ between ALL and AML through the B-ALL majority, so "truly DE" — and the base rate — are slightly
underestimated.) Yet
selection outside the folds turned exactly these chance genes into an 82 %-accurate "classifier".

⚠️ **Everything that looks at the labels belongs inside the folds:** gene filtering by t-test or fold change, choosing
the number of genes, choosing $k$, $\lambda$ or $C$, even removing "outlier" samples on the basis of the labels.
Label-free steps (e.g. log-transforming) can stay outside, though fitting a standardisation or a PCA on training folds
only is the safest habit.

✏️ **Your turn.** In 6.1, keep 100 genes instead of 10. Does the selection bias on pure noise grow or shrink? Then switch
to LOOCV (`kfold_indices(72, 72, rng)`) with selection outside: is LOOCV immune?

---
## 7 · Regularization, with λ chosen by cross-validation

> 📍 **Where we are.** The second weapon. In [Module 07](07_ridge_lasso_sparse_regression.ipynb) you met ridge and lasso;
> here they become a model-selection tool in which the model's complexity is a continuous knob.

Maximum likelihood always prefers the more complex model — complex models always have a better data log-likelihood
(variance ↑) — so we penalise complexity:

$$\min_\theta\ -\log P(D\mid\theta)+\lambda F(\theta),\qquad F(\theta)=\sum_{j=1}^{J}|\theta_j|\ \ (\text{L1, lasso}).$$

*In words:* trade goodness of fit against a cost for large (or many nonzero) parameters. $\lambda$ is the exchange rate:
large $\lambda$ = strong penalty = simpler model (**bias ↑**); small $\lambda$ = weak penalty (**variance ↑**). "Model
selection = select λ" — by cross-validation, whose downside is that it is time-consuming and trains on smaller
data sets. For Gaussian noise, $-\log P(D\mid\theta)$ is the residual sum of squares up to constants, so this is the lasso.

We apply it to the prostate-cancer study of Modules 06–07 (Stamey *et al.* 1989; as there, its simulated look-alike,
whose true coefficients are those the original study gave): predict log-PSA from 8 clinical measurements
in 67 training men, choose $\lambda$ by 10-fold CV, and test once on the 30 held-out men.

In [ ]:
# ── 7.1 Lasso on the prostate data (SIMULATED look-alike): 10-fold CV over lambda, then one look at the test set ──
pro = simulate_prostate_clinical()
features = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
tr_mask = pro["train"].to_numpy()
Xp_tr, mu_p, sd_p = standardize(pro.loc[tr_mask, features].to_numpy())       # scale with TRAINING men only
Xp_te = standardize(pro.loc[~tr_mask, features].to_numpy(), mu_p, sd_p)[0]
yp_tr, yp_te = pro.loc[tr_mask, "lpsa"].to_numpy(), pro.loc[~tr_mask, "lpsa"].to_numpy()
alphas = np.exp(np.linspace(np.log(1.0), np.log(1e-3), 40))                   # sklearn's alpha = our λ / (2N)
folds_p = kfold_indices(len(yp_tr), 10, np.random.default_rng(SEED + 7))
cv_mse, cv_se = [], []
for a in alphas:
    fold_mse = []
    for fold in folds_p:
        train = np.setdiff1d(np.arange(len(yp_tr)), fold)
        m = Lasso(alpha=a).fit(Xp_tr[train], yp_tr[train])
        fold_mse.append(np.mean((m.predict(Xp_tr[fold]) - yp_tr[fold]) ** 2))
    cv_mse.append(np.mean(fold_mse)); cv_se.append(np.std(fold_mse, ddof=1) / np.sqrt(len(folds_p)))
cv_mse, cv_se = np.array(cv_mse), np.array(cv_se)
i_min = int(np.argmin(cv_mse))
i_1se = int(np.flatnonzero(cv_mse <= cv_mse[i_min] + cv_se[i_min])[0])          # largest alpha within 1 SE (alphas decrease)
for name, i in [("minimum CV error", i_min), ("one-standard-error rule", i_1se)]:
    m = Lasso(alpha=alphas[i]).fit(Xp_tr, yp_tr)
    kept = [f for f, c in zip(features, m.coef_) if abs(c) > 1e-10]
    print(f"{name:24s}: alpha = {alphas[i]:.4f} | CV MSE {cv_mse[i]:.3f} | test MSE {np.mean((m.predict(Xp_te) - yp_te) ** 2):.3f} | "
          f"{len(kept)} predictors: {', '.join(kept)}")
ols = np.linalg.lstsq(np.c_[np.ones(len(yp_tr)), Xp_tr], yp_tr, rcond=None)[0]
print(f"{'least squares (all 8)':24s}: test MSE {np.mean((np.c_[np.ones(len(yp_te)), Xp_te] @ ols - yp_te) ** 2):.3f}  (ESL Table 3.3: 0.521)")

In [ ]:
# ── 7.2 The CV curve and the coefficient path ──
path = np.array([Lasso(alpha=a).fit(Xp_tr, yp_tr).coef_ for a in alphas])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.errorbar(np.log(alphas), cv_mse, yerr=cv_se, fmt="o-", ms=3, color="tab:purple", ecolor="0.7", label="10-fold CV ± 1 SE")
a1.axvline(np.log(alphas[i_min]), color="k", ls=":", label="minimum")
a1.axvline(np.log(alphas[i_1se]), color="tab:green", ls="--", label="1-SE rule")
a1.set_xlabel("log(alpha)   ← more penalty | less penalty →"); a1.set_ylabel("CV mean squared error")
a1.invert_xaxis(); a1.set_title("choosing λ by CV (SIMULATED prostate data, 67 men)"); a1.legend(fontsize=7)
for j, f in enumerate(features):
    a2.plot(np.log(alphas), path[:, j], label=f)
a2.axvline(np.log(alphas[i_min]), color="k", ls=":"); a2.axvline(np.log(alphas[i_1se]), color="tab:green", ls="--")
a2.invert_xaxis(); a2.set_xlabel("log(alpha)"); a2.set_ylabel("coefficient (standardised)")
a2.set_title("lasso path: predictors enter one by one"); a2.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.show()
entry = np.argmax(np.abs(path) > 1e-10, axis=0)                  # first alpha (from large to small) with a nonzero coefficient
print("order in which predictors enter as the penalty decreases:", [features[j] for j in np.argsort(entry, kind="stable")])

🔍 **What just happened?** Reading the CV curve from left (strong penalty) to right (weak penalty): with a huge $\lambda$
all coefficients are zero and the error is large (high bias); as $\lambda$ decreases, predictors enter one at a time
(`lcavol` first, then `lweight`, `svi` and `pgg45`, then `age`, `lbph`, `lcp`; `gleason` last) and the CV error drops;
at small $\lambda$ the curve is flat — the last predictors neither help nor hurt much with 67 men. The **one-standard-error rule** (Hastie *et al.*
2009, §7.10) picks the simplest model whose CV error is within one standard error of the minimum: it trades a
statistically invisible loss of fit for a sparser, more stable model. Here the CV minimum is at the smallest penalty of
the grid and keeps all 8 predictors (test MSE 0.351, the same as least squares, 0.348); the 1-SE rule keeps 4 (`lcavol`,
`lweight`, `svi`, `pgg45`) and does clearly worse on the 30 untouched test men (0.579). 🧪 The simulation explains why:
these data were generated with all eight coefficients non-zero (Module 06), three of them (`age`, `lbph`, `lcp`) of size
0.14–0.29, so a 4-predictor model is too rigid — high bias. The 1-SE rule is a sensible default when you value
simplicity or expect a sparse truth, but it is a bias–variance bet, not a free lunch. (On the original data ESL found the
sparser lasso models predicting at least as well as least squares, test MSE 0.479 vs 0.521.)

⚠️ **Selected predictors are not proven causes.** The lasso keeps a predictor because it helps prediction *given the
others*; correlated predictors can swap in and out between CV folds, and a dropped predictor may still be biologically
important.

### 7.3 Bias² + variance along λ

A classic figure (Bishop 2006, Fig. 3.6) shows that bias² + variance predicts (the shape of) the test error quite well as
$\ln\lambda$ varies. It was computed for the *quadratic* (ridge) penalty on 100 data sets of 25 points drawn from
$\sin(2\pi x)$ and fitted with 24 Gaussian basis functions, and the same message holds for the L1 penalty. We simulate
this setting for **both** penalties.

In [ ]:
# ── 7.3 Bias², variance and test error along lambda, for ridge and lasso (SIMULATED, 100 data sets) ──
def gaussian_basis(x, n_basis=24, width=0.1):
    """Features exp(−(x − μ_j)² / (2 width²)) for 24 centres μ_j evenly spaced on [0, 1]."""
    centres = np.linspace(0, 1, n_basis)
    return np.exp(-(np.asarray(x)[:, None] - centres) ** 2 / (2 * width ** 2))


rng = np.random.default_rng(SEED + 73)
sets = [(x, circadian(x) + rng.normal(0, SIGMA, 25)) for x in rng.uniform(0, 1, (100, 25))]
x_eval = np.linspace(0, 1, 200)
B_eval = gaussian_basis(x_eval)
curves = {}
for kind, grid in [("ridge", np.exp(np.linspace(-3, 2, 11))), ("lasso", np.exp(np.linspace(-8, -2, 13)))]:
    out = []
    for lam in grid:
        P = []
        for x, y in sets:
            B = gaussian_basis(x)
            if kind == "ridge":                                        # closed form with an unpenalised intercept
                Bc, yc = B - B.mean(0), y - y.mean()
                w = np.linalg.solve(Bc.T @ Bc + lam * np.eye(B.shape[1]), Bc.T @ yc)
                P.append(y.mean() + (B_eval - B.mean(0)) @ w)
            else:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    P.append(Lasso(alpha=lam, max_iter=20000).fit(B, y).predict(B_eval))
        P = np.array(P)
        b2, v = np.mean((P.mean(0) - circadian(x_eval)) ** 2), np.mean(P.var(0))
        y_fresh = circadian(x_eval) + rng.normal(0, SIGMA, P.shape)
        out.append((np.log(lam), b2, v, np.mean((y_fresh - P) ** 2) - SIGMA ** 2))
    curves[kind] = np.array(out)
print({kind: c.shape for kind, c in curves.items()}, "(rows = penalty values; columns = ln, bias², variance, test − σ²)")

In [ ]:
# ── 7.4 Plot the four curves for each penalty ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, kind in zip(axes, ["ridge", "lasso"]):
    c = curves[kind]
    ax.plot(c[:, 0], c[:, 1], color="tab:blue", label="bias²")
    ax.plot(c[:, 0], c[:, 2], color="tab:red", label="variance")
    ax.plot(c[:, 0], c[:, 1] + c[:, 2], color="tab:pink", lw=2, label="bias² + variance")
    ax.plot(c[:, 0], c[:, 3], "k--", label="test error − σ²")
    ax.set_xlabel("ln λ" if kind == "ridge" else "ln α (sklearn's lasso penalty)"); ax.set_ylabel("error")
    ax.set_title(f"{kind}: 100 SIMULATED data sets of 25 points"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
for kind in curves:
    c = curves[kind]
    print(f"{kind}: bias²+variance is smallest at ln = {c[np.argmin(c[:, 1] + c[:, 2]), 0]:.1f}; "
          f"test error is smallest at ln = {c[np.argmin(c[:, 3]), 0]:.1f}")

🔍 **What just happened?** For both penalties, increasing $\lambda$ trades variance (red, falling) for bias² (blue,
rising); their sum (pink) has a minimum, and the directly simulated test error (dashed, shifted down by the noise
$\sigma^2=0.09$ so the curves are comparable) lies on top of it — with the minimum at (nearly) the same $\lambda$. That
is the message "bias² + variance predicts the shape of the test error", now checked for the lasso *and* for ridge. In real life we cannot compute bias² and variance
(they need the true $f$ and many data sets), which is exactly why we use cross-validation to locate the minimum.

✏️ **Your turn.** In 7.1, change the fold assignment (`SEED + 7` → `SEED + 70`) and use 5 instead of 10 folds. Does the
chosen alpha move? Does the one-standard-error choice move? What does this tell you about reporting "the" selected genes
or predictors of a lasso model?

---
## 8 · The Bayesian information criterion

> 📍 **Where we are.** The third weapon needs no held-out data at all: it corrects the training
> log-likelihood for the number of parameters.

$$\mathrm{BIC}(M)=-2\log P\big(D\mid\hat\theta_M\big)+k_M\log n .$$

*In words:* take the best (maximum-likelihood) fit $\hat\theta_M$ of model $M$, measure its misfit as $-2$ times its
log-likelihood, and add a penalty of $\log n$ for each of its $k_M$ free parameters ($n$ = number of samples). Choose the
model with the **smallest** BIC. Larger $k$, higher penalty: more complex models are penalised.

⚠️ **A notational subtlety.** BIC is often written $\min_\theta\,-2\log P(D\mid\theta)+k\log n$. For a fixed model, $k$ is a
constant, so minimising over $\theta$ just gives the MLE; the minimisation that does the *selecting* is over **models**
(e.g. polynomial degrees), each evaluated at its own MLE.

**Where it comes from.** Schwarz (1978) showed that for large $n$, $-2\log P(D\mid M)\approx\mathrm{BIC}(M)$, where
$P(D\mid M)=\int P(D\mid\theta,M)P(\theta\mid M)\,d\theta$ is the **marginal likelihood** of the model. So choosing the
smallest BIC approximates choosing the model with the highest posterior probability under equal prior odds — Bayesian
model selection ([Module 21](21_bayesian_learning_mcmc.ipynb)) without doing the integral.

For polynomial regression with Gaussian noise, plugging in the MLE $\hat\sigma^2=\mathrm{RSS}/n$ gives
$-2\log P(D\mid\hat\theta)=n\log(2\pi\hat\sigma^2)+n$, with $k=(d+1)+1$ parameters ($d+1$ coefficients and $\sigma^2$).

In [ ]:
# ── 8.1 BIC vs 10-fold CV for choosing the polynomial degree (200 SIMULATED data sets of 30 points) ──
def bic_poly(t, y, degree):
    """BIC of a degree-`degree` polynomial with Gaussian noise, evaluated at the maximum-likelihood fit."""
    n = len(y)
    rss = np.sum((predict_poly(fit_poly(t, y, degree), t) - y) ** 2)
    return n * np.log(2 * np.pi * rss / n) + n + (degree + 2) * np.log(n)


rng = np.random.default_rng(SEED + 8)
choice = {"BIC": [], "10-fold CV": [], "true test error (oracle)": []}
for r in range(200):
    t_r = rng.uniform(0, 1, 30)
    y_r = circadian(t_r) + rng.normal(0, SIGMA, 30)
    choice["BIC"].append(np.argmin([bic_poly(t_r, y_r, d) for d in degrees]))
    choice["10-fold CV"].append(np.argmin([cv_mse_poly(t_r, y_r, d, 10, rng) for d in degrees]))
    choice["true test error (oracle)"].append(
        np.argmin([np.mean((predict_poly(fit_poly(t_r, y_r, d), t_new) - y_new) ** 2) for d in degrees]))
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (name, ch) in enumerate(choice.items()):
    counts = np.bincount(ch, minlength=10)
    ax.bar(degrees + (i - 1) * 0.27, counts, width=0.27, label=name)
    print(f"{name:26s}: chosen degree counts {dict((int(d), int(c)) for d, c in enumerate(counts) if c)}")
ax.set_xticks(degrees); ax.set_xlabel("chosen polynomial degree"); ax.set_ylabel("number of data sets (of 200)")
ax.set_title("which degree does each criterion pick? (SIMULATED)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For each of 200 data sets, the oracle (which knows the true test error) picks degree 3 most
often, sometimes 4–6. **BIC** picks degree 3 even more consistently: its $\log n$ penalty favours the smallest adequate
model. **CV** spreads its choices more widely (its estimates are noisy, Section 5.4) and occasionally picks a high degree.
Neither criterion is uniformly better: BIC is consistent — with growing $n$ it finds the true model *if* the true model
is among the candidates — whereas CV targets prediction error directly and needs no likelihood. BIC requires a
likelihood and a parameter count, which is easy for polynomials, mixtures and HMMs and awkward for k-NN or random forests.

BIC is the standard way to choose the **number of mixture components** ([Module 13](13_mixture_models_em.ipynb)) and
**HMM states** ([Module 20](20_hmm_learning.ipynb)). A quick example with a simulated marker gene measured in single
cells from three populations:

In [ ]:
# ── 8.2 How many cell populations? BIC for Gaussian mixtures (SIMULATED single-cell marker expression) ──
rng = np.random.default_rng(SEED + 82)
expr = np.r_[rng.normal(2.0, 0.6, 150), rng.normal(4.5, 0.7, 100), rng.normal(7.0, 0.5, 50)]   # 3 populations
bics = []
for K in range(1, 7):
    gm = GaussianMixture(n_components=K, n_init=3, random_state=SEED).fit(expr[:, None])
    bics.append(gm.bic(expr[:, None]))
    print(f"K = {K}: −2 log L = {-2 * gm.score(expr[:, None]) * len(expr):8.1f} | k = {3 * K - 1:2d} parameters | BIC = {bics[-1]:8.1f}")
K_best = int(np.argmin(bics)) + 1
gm = GaussianMixture(n_components=K_best, n_init=3, random_state=SEED).fit(expr[:, None])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
a1.plot(range(1, 7), bics, "o-"); a1.set_xlabel("number of mixture components K"); a1.set_ylabel("BIC")
a1.set_title(f"BIC is smallest at K = {K_best}")
xs = np.linspace(0, 9, 400)
a2.hist(expr, bins=40, density=True, color="0.8", edgecolor="0.6")
a2.plot(xs, np.exp(gm.score_samples(xs[:, None])), color="tab:red", label=f"fitted {K_best}-component mixture")
a2.set_xlabel("marker log-expression (SIMULATED)"); a2.set_ylabel("density"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The log-likelihood keeps improving as components are added (more parameters always fit
better), but beyond $K=3$ the improvement no longer pays the $\log n\approx5.7$ penalty per extra parameter, and BIC
selects the three simulated populations. A 1-D mixture with $K$ components has $3K-1$ free parameters ($K$ means, $K$
variances, $K-1$ free weights). For an HMM, plug the parameter count of cell 3.3 into the same formula.

✏️ **Your turn.** Move the second population's mean from 4.5 to 3.0 in 8.2 so that it overlaps the first. Which $K$
does BIC choose now? Is that a failure of BIC, or a sensible answer given the data?

### 📘 Lecture notes — regularization and the Bayesian information criterion

- **Why penalise.** Complex models always reach a better training log-likelihood (variance ↑), so maximum likelihood alone
  overfits.
- **Regularization:** $\min_\theta\ -\log P(D\mid\theta) + \lambda F(\theta)$, e.g. $F(\theta) = \sum_j|\theta_j|$ (L1).
  *In words:* goodness of fit minus a price for complexity. Large $\lambda$ = more penalisation: $\lambda\uparrow$ ⇒
  bias ↑, $\lambda\downarrow$ ⇒ variance ↑. **Model selection = select $\lambda$**, by cross-validation (cons:
  time-consuming; each fit uses less training data).
- **Bias² + variance along $\lambda$** predicts the shape of the test error quite well (Section 7.3).
- **BIC:** $\mathrm{BIC}(M) = -2\log P(D\mid\hat\theta_M) + k\log n$, with $k$ free parameters and $n$ samples; choose the
  smallest. Larger $k$, higher penalty — complex models are penalised; no held-out data needed. For large $n$ it
  approximates $-2\log P(D\mid M)$, the Bayesian model evidence.
- **Worked example.** $n = 100$ ($\ln 100 = 4.61$): model A ($k = 3$, $-2\log L = 50.0$) has BIC $= 50 + 3\cdot4.61 = 63.8$;
  model B ($k = 6$, $-2\log L = 40.0$) has $40 + 6\cdot4.61 = 67.6$ → choose A: B's better fit does not pay for three
  extra parameters.

> 🎓 **Exam tip.** Be able to say which direction of $\lambda$ increases bias and which increases variance, why $\lambda$ must not be chosen on the test set, and to compute and compare BIC values by hand.

---
## 9 · The Bayes error rate: the limit of prediction

> 📍 **Where we are.** Bias, variance and model selection explain why a model can be *worse than necessary*. The last
> theoretical idea adds a floor: some problems cannot be solved perfectly by any method.

If we knew the true class-conditional densities $p(x\mid c)$ and prior probabilities $P(c)$, the best possible rule would
assign each $x$ to the class with the largest posterior probability $P(c\mid x)\propto p(x\mid c)P(c)$
([Module 08](08_knn_naive_bayes.ipynb)). Its error rate is the **Bayes error rate** — a fundamental performance limit: a lower
bound on the classification error of *any* algorithm on a given problem, i.e. the error rate of the optimal decision rule.
For two classes:

$$\varepsilon_{\text{Bayes}}=\int\min\big\{p(x\mid c{=}1)P(c{=}1),\ p(x\mid c{=}2)P(c{=}2)\big\}\,dx .$$

*In words:* at every $x$ the optimal rule picks the class whose weighted density is higher, and it is wrong exactly when
the sample actually came from the *other* class — which happens with probability proportional to the lower curve. The
Bayes error is therefore the area under the lower of the two weighted density curves: the overlap area, shaded blue in the
figure below.

⚠️ **Two details that are easy to get wrong when drawing this picture.** (1) The bell-shaped curves are weighted
class-conditional densities $p(x\mid c_i)P(c_i)$, not posteriors $P(c_i\mid x)$ (posteriors would add up to 1 at every $x$). (2) In the "hard problem"
the two classes have different spreads; then the optimal rule has **two** decision boundaries, because far out in
*either* tail the wider class is more likely.

In [ ]:
# ── 9.1 Bayes error of a simple and a hard problem, by numerical integration ──
PROBLEMS = {"simple problem": dict(m1=-1.5, s1=1.0, m2=1.5, s2=1.0),
            "hard problem": dict(m1=0.0, s1=0.6, m2=0.6, s2=1.3)}
xs = np.linspace(-8, 8, 160001)
dx = xs[1] - xs[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (name, pr) in zip(axes, PROBLEMS.items()):
    g1 = 0.5 * stats.norm.pdf(xs, pr["m1"], pr["s1"])                 # p(x | c=1) P(c=1), equal priors
    g2 = 0.5 * stats.norm.pdf(xs, pr["m2"], pr["s2"])
    pr["bayes"] = np.sum(np.minimum(g1, g2)) * dx
    cuts = xs[np.flatnonzero(np.diff((g1 > g2).astype(int)) != 0)]       # where the larger curve switches
    pr["cuts"] = cuts
    ax.plot(xs, g1, color="tab:red", label="class 1: $p(x|c=1)P(c=1)$")
    ax.plot(xs, g2, color="tab:blue", label="class 2: $p(x|c=2)P(c=2)$")
    ax.fill_between(xs, np.minimum(g1, g2), color="tab:blue", alpha=0.35, label="Bayes error area")
    for c in cuts:
        ax.axvline(c, color="tab:green", lw=1.5)
    ax.set_xlim(-5, 5); ax.set_xlabel("input x"); ax.set_title(f"{name}: Bayes error {pr['bayes']:.4f}", fontsize=10)
    print(f"{name}: Bayes error = {pr['bayes']:.4f}; optimal decision boundaries at x = {np.round(cuts, 3) + 0.0}")
axes[0].set_ylabel("weighted density"); axes[0].legend(fontsize=7)
print("check (equal variances): Phi(−Δ/2σ) =", round(stats.norm.cdf(-1.5), 4))
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the simple problem (equal spreads, means 3 SDs apart) the Bayes error is 6.7 %, matching
the closed form $\Phi(-\Delta/2\sigma)=\Phi(-1.5)$, with one boundary midway between the means. For the hard problem the
curves overlap heavily: even the optimal rule is wrong 29 % of the time, and it needs two boundaries (green): class 2 wins
on the far left as well as on the right. No amount of data or cleverness can do better *with this input*; to improve,
one must measure something else (a new feature, a better assay).

Can real classifiers reach the floor? We train k-NN ($k=\sqrt n$) and logistic regression on growing training sets and
compute each trained rule's **exact** error by integrating over $x$ (a finite test set would add its own sampling noise
and could even dip below the Bayes error by chance).

In [ ]:
# ── 9.2 k-NN and logistic regression approach, but never beat, the Bayes error ──
def exact_error(predict, pr):
    """Error rate of a 1-D classifier: integrate the weighted density of the class it does NOT predict."""
    pred = predict(xs[::20, None])                                     # a 0.002-spaced grid
    g1 = 0.5 * stats.norm.pdf(xs[::20], pr["m1"], pr["s1"])
    g2 = 0.5 * stats.norm.pdf(xs[::20], pr["m2"], pr["s2"])
    return np.sum(np.where(pred == 1, g1, g2)) * dx * 20               # label 1 means "class 2"


rng = np.random.default_rng(SEED + 9)
sizes = [20, 100, 1000, 10000]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharex=True)
for ax, (name, pr) in zip(axes, PROBLEMS.items()):
    errs = {"logistic regression": [], "k-NN (k = √n)": []}
    for n in sizes:
        lab = rng.integers(0, 2, n)                                      # 0 = class 1, 1 = class 2
        x = np.where(lab == 0, rng.normal(pr["m1"], pr["s1"], n), rng.normal(pr["m2"], pr["s2"], n))[:, None]
        errs["logistic regression"].append(exact_error(LogisticRegression().fit(x, lab).predict, pr))
        errs["k-NN (k = √n)"].append(exact_error(KNeighborsClassifier(int(np.sqrt(n))).fit(x, lab).predict, pr))
    for (lab_name, e), col in zip(errs.items(), ["tab:orange", "tab:purple"]):
        ax.plot(sizes, e, "o-", color=col, label=lab_name)
        print(f"{name:15s} {lab_name:20s}: " + "  ".join(f"n={n}: {v:.4f}" for n, v in zip(sizes, e)))
    ax.axhline(pr["bayes"], color="tab:blue", ls="--", label="Bayes error")
    ax.set_xscale("log"); ax.set_xlabel("training-set size n"); ax.set_title(name)
axes[0].set_ylabel("exact error rate"); axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every error rate lies above the dashed Bayes line, as it must. On the simple problem both
methods get close quickly: the optimal rule is a single threshold, which logistic regression can represent exactly. On
the hard problem k-NN creeps down towards 0.29 as $n$ grows (it can learn two thresholds), but logistic regression
stalls well above it whatever the sample size: a linear model can only place *one* threshold, so it carries a **bias**
that no amount of data removes. The gap between a method's error and the Bayes error is the part that model selection
and more data can attack; the Bayes error itself cannot be attacked with the same features.

✏️ **Your turn.** Give logistic regression the feature $x^2$ as well (`np.c_[x, x**2]`; adapt `exact_error` to pass
`np.c_[g, g**2]` for the grid points `g`). Does it now reach the Bayes error on the hard problem? Why?

### 📘 Lecture notes — the Bayes error rate, and the lecture in one list

- **Definition.** A fundamental performance limit for classification: a lower bound on the error of *any* algorithm on a
  given problem, i.e. the error rate of the optimal decision rule (assign $x$ to the class with the largest
  $p(x\mid c)P(c)$).
- **Formula (two classes):** $\varepsilon_{\text{Bayes}} = \int\min_c\, p(x\mid c)P(c)\,dx$. *In words:* the overlap area
  under the lower of the two weighted class densities.
- **Simple vs hard problem.** Well-separated classes give a small overlap; heavily overlapping classes a large Bayes
  error that no algorithm or amount of data can beat with the same features. Equal-variance Gaussians with equal priors:
  $\varepsilon = \Phi(-\Delta/2\sigma)$ (e.g. means 3 SDs apart → 6.7 %).
- **Summary of model selection.** (1) The bias–variance trade-off. (2) The battle against overfitting: cross-validation,
  the Bayesian information criterion, regularization. (3) The Bayes error rate as the floor.

> 🎓 **Exam tip.** Be able to compute the Bayes error for two equal-variance Gaussians ($\Phi(-\Delta/2\sigma)$ with equal priors) and to explain why a training or test error *below* the Bayes error signals overfitting or a too-small test set.

---
## 10 · Case study: classifying leukaemias with nested cross-validation

> 📍 **Where we are.** Everything comes together. The 02-620 lecture closes with a case study: 72 leukaemia samples
> (47 ALL, 25 AML) measured on a microarray with about 7,130 genes, and three classifiers — k-NN, a Gaussian classifier
> and logistic regression — with training and test error plotted against the number of features used.

The widely used preprocessed version of the Golub data (Dudoit, Fridlyand & Speed 2002) keeps 3,571 genes after removing
probes with low or unreliable expression (the original Affymetrix data have 7,129 probes). Our simulated look-alike has
the same shape: 72 samples × 3,571 genes, Golub's original 38/34 training/test design, a detection floor, and 54
named marker genes (CST3, ZYX, MPO, … for AML; MYB, TCF3, CD79A, … for ALL).
The three classifiers, all written from scratch (and all with the gene ranking done on the training samples only):

* **k-NN** with $k=3$ ([Module 08](08_knn_naive_bayes.ipynb)) on standardised genes;
* a **Gaussian classifier** — a Gaussian generative model: each class is modelled by independent Gaussians
  per gene (class-specific means and variances, i.e. Gaussian naïve Bayes; Modules 08–09);
* **logistic regression** with a small L2 penalty, fitted by gradient descent ([Module 09](09_logistic_regression.ipynb)).
  Without a penalty the coefficients would diverge: with more genes than samples, the classes are perfectly separable.

In [ ]:
# ── 10.1 Three classifiers, each 'fit on train, predict test' ──
def knn3(Xtr, ytr, Xte):
    """3-nearest-neighbour classifier."""
    return knn_predict(Xtr, ytr, Xte, k=3)


def gaussian_classifier(Xtr, ytr, Xte):
    """Gaussian generative classifier with independent genes (class-specific means and variances)."""
    log_post = []
    for c in (0, 1):
        A = Xtr[ytr == c]
        m, v = A.mean(axis=0), A.var(axis=0) + 1e-3                         # small floor avoids zero variances
        log_post.append(np.log(len(A) / len(ytr)) - 0.5 * np.sum(np.log(2 * np.pi * v) + (Xte - m) ** 2 / v, axis=1))
    return (log_post[1] > log_post[0]).astype(int)


def logistic_l2(Xtr, ytr, Xte, lam=0.1, lr=0.5, iters=500):
    """Logistic regression with penalty (lam/2)·||w||², fitted by gradient descent."""
    w, b = np.zeros(Xtr.shape[1]), 0.0
    for _ in range(iters):
        p = sigmoid(Xtr @ w + b)
        w -= lr * (Xtr.T @ (p - ytr) / len(ytr) + lam * w)
        b -= lr * np.mean(p - ytr)
    return (Xte @ w + b > 0).astype(int)


CLASSIFIERS = {"kNN (k=3)": knn3, "Gaussian classifier": gaussian_classifier, "logistic regression (L2)": logistic_l2}


def select_fit_predict(clf, Xtr, ytr, Xte, m):
    """Rank genes on the training samples, keep the top m, standardise with training statistics, fit and predict."""
    genes = top_genes(Xtr, ytr, m)
    Ztr, mu, sd = standardize(Xtr[:, genes])
    return clf(Ztr, ytr, standardize(Xte[:, genes], mu, sd)[0])


tr_g = golub["is_train"]
print(f"Golub's original split: {tr_g.sum()} training samples ({yg[tr_g].sum()} AML), "
      f"{(~tr_g).sum()} test samples ({yg[~tr_g].sum()} AML)")

In [ ]:
# ── 10.2 Training and test error vs number of genes (original 38/34 split) ──
n_genes = np.r_[1:21, 25:101:5]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharey=True)
for ax, (name, clf) in zip(axes, CLASSIFIERS.items()):
    tr_e = [np.mean(select_fit_predict(clf, Xg[tr_g], yg[tr_g], Xg[tr_g], m) != yg[tr_g]) for m in n_genes]
    te_e = [np.mean(select_fit_predict(clf, Xg[tr_g], yg[tr_g], Xg[~tr_g], m) != yg[~tr_g]) for m in n_genes]
    ax.plot(n_genes, tr_e, ":", color="tab:blue", label="training error")
    ax.plot(n_genes, te_e, "-", color="tab:blue", label="testing error")
    ax.set_title(name, fontsize=10); ax.set_xlabel("number of features (top-|t| genes)")
    print(f"{name:26s}: test error with 1, 5, 10, 20, 50, 100 genes = "
          + ", ".join(f"{te_e[list(n_genes).index(m)]:.3f}" for m in [1, 5, 10, 20, 50, 100]))
axes[0].set_ylabel("error rate"); axes[0].set_ylim(-0.02, 0.5); axes[0].legend(fontsize=7)
fig.suptitle(f"{golub['source']} — 38 training / 34 test samples", fontsize=9)
plt.tight_layout(); plt.show()
ids_te, atyp_te, lab_te = golub["sample_id"][~tr_g], golub["atypical"][~tr_g], golub["labels"][~tr_g]
for m in [10, 50]:                                                    # who is misclassified?
    for name, clf in CLASSIFIERS.items():
        wrong = select_fit_predict(clf, Xg[tr_g], yg[tr_g], Xg[~tr_g], m) != yg[~tr_g]
        who = [f"#{i} {lab}" + (" (atypical)" if at else "") for i, lab, at in zip(ids_te[wrong], lab_te[wrong], atyp_te[wrong])]
        print(f"{m:3d} genes, {name:26s}: misclassified test samples: {', '.join(who) or 'none'}")

🔍 **What just happened?** Training errors (dotted) are zero from the very first genes for all three classifiers — with
38 samples, almost any rule fits the training set. The test errors on the 34 independent samples (solid) move in steps of
$1/34\approx0.03$ (one patient) and depend on the number of genes in a jagged way: k-NN and logistic regression make two
errors with few genes and none from about 20 (k-NN) or 35 (logistic regression) genes on; the Gaussian classifier stays at
two or three errors whatever the number of genes. *Who* is misclassified? With 10 genes all three classifiers miss the
same two test samples, #47 and #67 — the two **atypical AML samples** planted in the simulation (their AML signature is
only about a third as strong; real data also contain such borderline cases). Many weakly informative genes together
rescue them for k-NN and logistic regression; the Gaussian classifier still misses both (and #71 with 50 genes). In
nested CV, where every model is trained on about 65 samples instead of 38 (about 22 AML instead of 11), the Gaussian
classifier gets all 72 samples right — plausibly because it must estimate a mean *and* a variance per gene for each
class, which makes it the most data-hungry of the three. The simulated ALL/AML signal is strong — as in Golub's data,
where a few dozen genes already separate the classes almost perfectly — so all errors are small.

Two problems remain. First, reading the best number of genes off this plot uses the **test** set for model selection.
Second, a single 38/34 split is a noisy yardstick (Golub's test set even came partly from different centres and tissue
sources). **Nested cross-validation** solves both: an *outer* CV loop estimates the error, and inside each outer training
set an *inner* CV loop chooses the number of genes — so the outer test fold never influences any choice.

```text
for each outer fold j = 1..10:                      # estimates the error of the WHOLE procedure
    outer_train, outer_test = all samples except fold j, fold j
    for each candidate m in {2, 5, 10, 20, 50, 100}:
        inner_score[m] = 5-fold CV error of "select top-m genes + classifier" on outer_train only
    m* = argmin inner_score                         # model selection sees outer_train only
    error_j = errors of "select top-m* genes on outer_train + classifier" on outer_test
nested CV error = total errors / 72
```

In [ ]:
# ── 10.3 Nested cross-validation for the three classifiers (SIMULATED Golub-like data) ──
def stratified_folds(y, k, rng):
    """k folds with (nearly) the same class proportions: deal each class's shuffled samples round-robin."""
    folds = [[] for _ in range(k)]
    for c in np.unique(y):
        for i, s in enumerate(rng.permutation(np.flatnonzero(y == c))):
            folds[i % k].append(s)
    return [np.array(sorted(f)) for f in folds]


def cv_error_clf(clf, X, y, m, folds):
    """CV error of 'top-m genes (ranked inside each fold) + classifier'."""
    wrong = 0
    for fold in folds:
        train = np.setdiff1d(np.arange(len(y)), fold)
        wrong += np.sum(select_fit_predict(clf, X[train], y[train], X[fold], m) != y[fold])
    return wrong / len(y)


GRID = [2, 5, 10, 20, 50, 100]
t0 = time.time()
nested = {name: [] for name in CLASSIFIERS}
for rep in range(3):                                                  # three different outer splits
    outer = stratified_folds(yg, 10, np.random.default_rng(100 + rep))
    for name, clf in CLASSIFIERS.items():
        wrong, chosen = 0, []
        for fold in outer:
            train = np.setdiff1d(np.arange(72), fold)
            inner = stratified_folds(yg[train], 5, np.random.default_rng(200 + rep))
            m_star = GRID[int(np.argmin([cv_error_clf(clf, Xg[train], yg[train], m, inner) for m in GRID]))]
            chosen.append(m_star)
            wrong += np.sum(select_fit_predict(clf, Xg[train], yg[train], Xg[fold], m_star) != yg[fold])
        nested[name].append((wrong / 72, chosen))
print(f"nested CV done in {time.time() - t0:.0f} s")
for name in CLASSIFIERS:
    errs = [e for e, _ in nested[name]]
    print(f"{name:26s}: nested-CV error {', '.join(f'{e:.3f}' for e in errs)} (three outer splits) | "
          f"genes chosen in split 1: {nested[name][0][1]}")

In [ ]:
# ── 10.4 The optimistic shortcut: pick the best grid value by (non-nested) CV and report its CV error ──
flat_folds = stratified_folds(yg, 10, np.random.default_rng(100))
print(f"{'classifier':26s}  CV error for m = {GRID}            min (optimistic)   nested (split 1)")
for name, clf in CLASSIFIERS.items():
    flat = [cv_error_clf(clf, Xg, yg, m, flat_folds) for m in GRID]
    print(f"{name:26s}  {' '.join(f'{e:.3f}' for e in flat)}      {min(flat):.3f}             {nested[name][0][0]:.3f}")
se = np.sqrt(0.02 * 0.98 / 72)
print(f"\nfor reference: one misclassified patient = {1 / 72:.3f}; standard error of a 2 % error rate with 72 samples ≈ {se:.3f}")

🔍 **What just happened?** All three classifiers misclassify at most two of the 72 patients in nested CV (errors of
0–3 %), and the number of genes chosen by the inner loop varies from fold to fold (mostly 20 or 50, occasionally 2, 5, 10 or 100) —
a sign that many gene numbers work about equally well. The Gaussian classifier is error-free in all three outer splits,
k-NN makes 0–2 errors and logistic regression 1–2, but every difference is at most two patients (0.028), comparable to
the standard error of about 0.016 of a 2 % error estimate: **with 72 samples these three classifiers cannot be told
apart**. The non-nested shortcut (choose the best $m$ by CV, then report that same CV error) is equal to or lower than
the nested estimate (for logistic regression 0.000 vs 0.014) — optimistic, because the minimum of six noisy numbers is
biased downwards (Varma & Simon 2006). Here the bias is small because the grid is small and the signal is strong; with
large grids (many $k$, $C$, $\gamma$, gene numbers …) and weak signals it can be large.

**The 02-620 model-selection workflow, in one paragraph.** Decide what you want to generalise to (new patients) and make
that the unit of every split. Lock away a test set if you can afford one. Put *every* data-dependent choice — feature
selection, the number of genes, $k$, $\lambda$, $C$, $\gamma$, the architecture — inside the training part of an
(inner) cross-validation. Report the outer (nested) estimate with its uncertainty, compare it with a trivial baseline
(here: always "ALL", 35 % error) and keep in mind the Bayes error: a perfect score on a few dozen patients is more often a
leak than a miracle.

✏️ **Your turn.** Enlarge the grid to `GRID = [1, 2, 3, 5, 8, 10, 15, 20, 30, 50, 75, 100, 200, 500]` and rerun 10.3–10.4.
Does the gap between the optimistic (non-nested) and the nested estimate grow? Why would you expect it to?

### 📘 Lecture notes — the leukaemia case study

- **The case.** 72 leukaemia samples — 47 ALL (type I) and 25 AML (type II) — measured on a microarray with about 7,130
  genes (Golub *et al.* 1999). Three classifiers: k-NN, a Gaussian (generative) classifier, logistic regression.
- **The figure to understand.** Training error (dotted) and testing error (solid) against the number of features
  used: training error drops to zero after a few genes, test error is jagged and can rise with many genes —
  model selection again (how many genes?).
- **Doing it honestly.** Rank genes inside each training set; choose the number of genes by an inner CV; estimate the
  error of the whole procedure with an outer CV (nested CV). Reading the best number of genes off a test-error curve is
  model selection on the test set.
- **Resolution.** With 72 samples one patient is 1.4 % of the error; differences of one or two patients between
  classifiers are within noise.
- **The same lecture also recalls** sparse (lasso) regression for GWAS (Module 07) and the regularization slogan
  $\lambda\uparrow$ bias ↑, $\lambda\downarrow$ variance ↑ (Section 7).

> 🎓 **Exam tip.** Explain why reading the best number of features off a test-error curve is model selection on the test set, and write the nested-CV loop in pseudo-code (which loop selects, which loop estimates).

## 📌 Key takeaways

- **Model selection ≠ parameter estimation.** Every method has choices that are not fitted by maximum likelihood —
  features, $k$, tree size, graph structure, number of clusters or states, architecture, and for SVMs $C$, the kernel
  and $\gamma$ (not "NA").
- **Bias–variance trade-off.** Rigid models have high bias (high training error); flexible models have high variance
  (test error ≫ training error). For squared error, $\mathbb E[(y-\hat f)^2]=\text{Bias}^2+\text{Variance}+\sigma^2$
  exactly; the noise $\sigma^2$ is irreducible.
- **Cross-validation** estimates test error from training data alone. k-fold costs $k$ fits; 2-fold is pessimistic;
  $k=5$–10 is the usual compromise and LOOCV suits tiny data sets. Folds must respect the biological unit
  (patients, not biopsies) — independence matters more than $k$.
- **The cardinal rule**: the test data must not influence any choice. Selecting genes on all samples before CV
  (selection bias) can turn pure noise into an 80 %-accurate "signature"; put every label-dependent step inside the folds
  and use **nested CV** to report the performance of a tuned procedure.
- **Regularization** turns complexity into a continuous knob λ (λ↑ bias ↑, λ↓ variance ↑), chosen by CV; bias² +
  variance tracks the test error along λ.
- **BIC** $=-2\log P(D\mid\hat\theta)+k\log n$ penalises parameters without held-out data and approximates Bayesian model
  selection; it tends to choose simpler models than CV.
- **The Bayes error rate** is the error of the optimal rule, a floor no algorithm can beat with the same features;
  methods with too little flexibility (e.g. a linear boundary for a two-threshold problem) stay above it however large $n$.

---
## ✅ Checkpoint

1. For k-NN, what is 'parameter estimation' and what is 'model selection'?
   <details><summary>Answer</summary>k-NN has no fitted parameters beyond storing the training data (it is non-parametric); choosing $k$ (and the distance and feature scaling) is model selection.</details>
2. A method has bias² = 0.02, variance = 0.05 and the noise variance is 0.09 at some $x$. What is its expected squared test error there? Which term would doubling the training set change?
   <details><summary>Answer</summary>$0.02+0.05+0.09=0.16$. Mainly the variance (for least squares it roughly halves); the noise is unchanged and the bias of a fixed model class does not depend on $n$.</details>
3. In the derivation, why is $\mathbb E[\epsilon\hat f]=0$?
   <details><summary>Answer</summary>The noise of the *new* measurement is independent of the training data that determine $\hat f$, so $\mathbb E[\epsilon\hat f]=\mathbb E[\epsilon]\mathbb E[\hat f]$, and $\mathbb E[\epsilon]=0$.</details>
4. With $N=72$ samples, how many model fits do 10-fold CV and LOOCV need? Which has the more pessimistic estimate, 2-fold or LOOCV, and why?
   <details><summary>Answer</summary>10 and 72. 2-fold: each model is trained on only 36 samples, so it is worse than a model trained on 72, and the estimate is biased upwards.</details>
5. Describe in pseudo-code the wrong and the right way to combine t-test gene selection with cross-validation.
   <details><summary>Answer</summary>Wrong: rank genes on all samples → keep top m → CV the classifier. Right: for each fold, rank genes on the training part only → keep top m → train → predict the held-out fold.</details>
6. $n=30$. Model A (degree 3, $k=5$) has $-2\log L=10.0$; model B (degree 5, $k=7$) has $-2\log L=4.0$. Which does BIC choose? ($\ln 30\approx3.40$)
   <details><summary>Answer</summary>BIC(A) $=10+5\cdot3.40=27.0$; BIC(B) $=4+7\cdot3.40=27.8$. BIC chooses A: B's better fit does not pay for two extra parameters.</details>
7. Two classes $\mathcal N(-1,1)$ and $\mathcal N(1,1)$ with equal priors. What is the Bayes error, and where is the optimal boundary?
   <details><summary>Answer</summary>Boundary at $x=0$; error $=\Phi(-1)\approx0.159$.</details>
8. Code reading: in cell 10.3, which loop chooses the number of genes and which loop estimates the error? What would go wrong if `inner` were built from all 72 samples?
   <details><summary>Answer</summary>The inner loop (`cv_error_clf` over `inner` folds of `train`) chooses $m^*$; the outer loop over `outer` folds estimates the error. Inner folds built from all 72 samples would let the outer test fold influence the choice of $m^*$ — selection bias again.</details>

## 🧠 Questions & discussion

1. A paper reports 100 % LOOCV accuracy for a 12-gene signature distinguishing two tumour subtypes in 40 patients; the genes were chosen from 20,000 by a t-test. What would you ask the authors, and what analysis would convince you?
2. Your data contain 200 cells from each of 5 mice, and you want a classifier that works on new mice. How do you build the CV folds? What if you only care about new cells from the same mice?
3. BIC tends to pick simpler models than CV. When would you prefer one over the other in a biological study (think of the number of cell types in a mixture versus the best predictor of drug response)?
4. The Bayes error of a diagnostic problem is 20 % with gene expression alone. What could reduce it? Can a better algorithm?
5. Why is the minimum of a CV curve over a large grid of hyperparameters an optimistic estimate, and how large do you expect the optimism to be when the grid has 2 values versus 2,000?

## 🚀 Bonus — going further

- Add the bootstrap ('.632' estimator) and compare it with 10-fold CV and LOOCV in the simulation of cell 5.4 (Hastie et al. 2009, §7.11).
- Repeat Section 10 with an RBF SVM and a random forest, tuning $C$/$\gamma$ and the number of features per split by nested CV — a preview of the [capstone](44_capstone_project.ipynb). Random forests also offer the out-of-bag error: see [DecisionTreesRandomForests](../../DecisionTreesRandomForests/).
- Cross-validation gives an average error; **conformal prediction** gives per-sample prediction sets with guaranteed coverage: [ConformalPrediction](../../ConformalPrediction/).
- Ridge and lasso with λ chosen by CV on real wheat genomic data: [RegularizedLinearRegression](../../RegularizedLinearRegression/).
- BIC for choosing the number of particle classes in cryo-electron tomography: Module 04 of the sister course [BiologyToResearch](../../../../cryoem/tutorial/BiologyToResearch/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 23 · Model selection
Input: SIMULATED circadian time course, GWAS genotypes, pure-noise and repeated-biopsy matrices, Gaussian classes;
       SIMULATED look-alikes of Golub leukaemia (72 × 3,571), prostate cancer (97 men), breast tumours (569)
Output: train/test error curves, bias²/variance tables, CV curves, selection-bias histograms, lasso path,
        BIC choices, Bayes error, nested-CV error of three leukaemia classifiers
Method: least-squares polynomials, k-fold/LOOCV/grouped CV from scratch, t-test gene ranking inside folds,
        lasso and RBF-SVM tuning by CV, BIC, numerical Bayes error, nested CV
Key equation: E[(y − f̂)²] = Bias² + Var + σ²;   BIC = −2 log P(D | θ̂) + k log n
Key parameters: polynomial degree, k (folds), number of genes m, λ (alpha), C and γ, K (components)
What changed when I modified … (σ, number of time points, genes kept, folds, grouping): …
One failure mode: (e.g. selection outside the folds gave ~80 % accuracy on pure noise; record-wise CV on biopsies)
Connection to the 02-620 topics (which idea of the lecture this illustrates):
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on model selection.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 7 (bias–variance, cross-validation, BIC, the one-standard-error rule, §7.10 'the wrong and right way to do cross-validation', §7.12 conditional vs expected test error) and Chapter 3 (lasso on the prostate data).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §5.3 (Bayesian model selection, BIC) and Chapter 6 (bias–variance trade-off, cross-validation).
- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — §3.2 (the bias–variance decomposition; Fig. 3.6).
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *Proc Natl Acad Sci USA* 99:6562–6566.
- Varma S., Simon R. (2006) Bias in error estimation when using cross-validation for model selection. *BMC Bioinformatics* 7:91.
- Simon R., Radmacher M. D., Dobbin K., McShane L. M. (2003) Pitfalls in the use of DNA microarray data for diagnostic and prognostic classification. *J Natl Cancer Inst* 95:14–18.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *J Am Stat Assoc* 97:77–87.
- Schwarz G. (1978) Estimating the dimension of a model. *Ann Stat* 6:461–464.
- Tibshirani R. (1996) Regression shrinkage and selection via the lasso. *J R Stat Soc B* 58:267–288.
- Stamey T. A. *et al.* (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *J Urol* 141:1076–1083.
- Street W. N., Wolberg W. H., Mangasarian O. L. (1993) Nuclear feature extraction for breast tumor diagnosis. *Proc SPIE* 1905:861–870 (the Wisconsin diagnostic breast-cancer data).
- Cortes C., Vapnik V. (1995) Support-vector networks. *Mach Learn* 20:273–297.
- Fortmann-Roe S. (2012) Understanding the bias–variance tradeoff. https://scott.fortmann-roe.com/docs/BiasVariance.html (the dartboard picture of bias and variance).

## ➡️ Next

You now have every tool of 02-620 — and the discipline to evaluate them honestly. That completes **Level 1**.

- **Level 2 starts here:** [Module 24 · Attention, transformers and self-supervised learning](24_attention_transformers.ipynb)
  turns the weighted-neighbour averaging of Module 08 and the PyTorch training loops of Module 22 into
  self-attention — the mechanism behind every modern language model — and opens the road to protein,
  genomic, single-cell and AI-driven research (Modules 24–43).
- **Following the 02-620 lectures only?** Jump to the capstone,
  **[Module 44 · Capstone: one dataset, three methods](44_capstone_project.ipynb)**, which walks through a complete
  02-620-style project, from the question and the evaluation contract to nested cross-validation and an honest report.